
# Notebook 03 — Channel Behavior Audit

## 3.1 Candidate Data Loading & Reconciliation

**Research Question:** Can we accurately retrieve the natural Future activity of all 995 multi-channel attack candidates?

**Objectives:**
- Reuse the candidate population identified in Notebook 01.
- Restrict analysis to users with at least two naturally active channels.
- Load eligible Future user-days and event counts for Email, USB, and HTTP.
- Verify candidate coverage and user-day consistency with the existing population profile.

**Constraints:**
- Frequency-based analysis only.
- Historical Baseline remains unchanged.
- No injection, scenario assignment, or new eligibility rules.
- No repetition of previously completed temporal and channel-rotation analyses.

**Expected Output:** A validated candidate profile and chronological Future daily activity dataset, ready for behavioral comparison.


In [2]:

# ============================================================
# 3.1 CANDIDATE DATA LOADING & RECONCILIATION
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

DATA_ROOT = (
    PROJECT_ROOT.parent
    / "Data_Engineering"
    / "Data_engineering"
    / "data"
)

PROFILE_PATH = (
    PROJECT_ROOT
    / "reports"
    / "01_User_Population_Audit"
    / "user_population_profile.parquet"
)

FUTURE_PATH = (
    DATA_ROOT
    / "features"
    / "behavior_features_r62_v1.parquet"
)

BASELINE_PATH = (
    DATA_ROOT
    / "features"
    / "user_baseline_r62_v1.parquet"
)

REPORT_DIR_03 = (
    PROJECT_ROOT
    / "reports"
    / "03_Channel_Behavior_Audit"
)

for path in [PROFILE_PATH, FUTURE_PATH, BASELINE_PATH]:
    assert path.is_file(), f"Missing source: {path}"

# ------------------------------------------------------------
# 2. LOAD EXISTING POPULATION PROFILE
# ------------------------------------------------------------

profile_columns = [
    "user",
    "has_valid_baseline",
    "future_workdays",
    "active_channel_count",
    "observed_channel_set",
    "email_days",
    "usb_days",
    "http_days"
]

user_profile = pd.read_parquet(
    PROFILE_PATH,
    columns=profile_columns
)

# Reuse the previously established candidate classification.
candidate_profile = (
    user_profile.loc[
        user_profile["active_channel_count"] >= 2
    ]
    .copy()
    .sort_values("user")
    .reset_index(drop=True)
)

assert len(candidate_profile) == 995
assert candidate_profile["user"].is_unique
assert candidate_profile["has_valid_baseline"].all()
assert candidate_profile["future_workdays"].gt(0).all()

candidate_users = set(candidate_profile["user"])

# ------------------------------------------------------------
# 3. LOAD FUTURE EVENT COUNTS ONLY
# ------------------------------------------------------------

CHANNELS = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count"
}

daily_columns = [
    "user",
    "day",
    *CHANNELS.values()
]

future_daily = pd.read_parquet(
    FUTURE_PATH,
    columns=daily_columns
)

candidate_daily = (
    future_daily.loc[
        future_daily["user"].isin(candidate_users)
    ]
    .copy()
)

candidate_daily["day"] = pd.to_datetime(
    candidate_daily["day"]
)

candidate_daily = (
    candidate_daily
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. CANDIDATE COVERAGE VALIDATION
# ------------------------------------------------------------

assert not candidate_daily.empty

assert set(candidate_daily["user"]) == candidate_users

assert not candidate_daily.duplicated(
    ["user", "day"]
).any()

assert candidate_daily["day"].between(
    "2011-01-01", "2011-06-01"
).all()

actual_days = candidate_daily.groupby("user").size()

expected_days = (
    candidate_profile.set_index("user")["future_workdays"]
)

pd.testing.assert_series_equal(
    actual_days.sort_index().astype("int64"),
    expected_days.sort_index().astype("int64"),
    check_names=False
)

for column in CHANNELS.values():
    assert candidate_daily[column].notna().all()
    assert candidate_daily[column].ge(0).all()

# ------------------------------------------------------------
# 5. CONCISE SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 03 — CANDIDATE DATA LOADING")
print("=" * 65)

print(f"Candidate users       : {len(candidate_profile):,}")
print(f"Eligible Future days  : {len(candidate_daily):,}")
print(f"Two-channel users     : {(candidate_profile['active_channel_count'] == 2).sum():,}")
print(f"Three-channel users   : {(candidate_profile['active_channel_count'] == 3).sum():,}")

print("\nFUTURE COVERAGE")
print("First day             :", candidate_daily["day"].min().date())
print("Last day              :", candidate_daily["day"].max().date())

print("\nVALIDATION")
print("Candidate population  : PASS")
print("User-day reconciliation: PASS")
print("Original sources      : READ ONLY")
print("Files written         : NONE")


NOTEBOOK 03 — CANDIDATE DATA LOADING
Candidate users       : 995
Eligible Future days  : 103,395
Two-channel users     : 704
Three-channel users   : 291

FUTURE COVERAGE
First day             : 2011-01-03
Last day              : 2011-06-01

VALIDATION
Candidate population  : PASS
User-day reconciliation: PASS
Original sources      : READ ONLY
Files written         : NONE



## 3.2 Historical Frequency Reference Audit

**Research Question:** Which of the 995 candidates have usable, precomputed historical frequency statistics for each channel?

**Objectives:**
- Read the existing Baseline frequency statistics without recalculating them.
- Inspect historical channel activity and the availability of P50, P75, P90, P95, and MAD.
- Reconcile the results with the channel-baseline availability flags from Notebook 01.
- Identify Future-active channels without usable historical references.

**Important:** Baseline percentiles and MAD were calculated using historical channel-active workdays only.

**Constraints:**
- Frequency/count metrics only.
- Read-only Historical Baseline.
- No automatic candidate exclusion.
- No new stealth thresholds or attack rules.

**Expected Output:** A per-user, per-channel Baseline reference audit and a concise availability summary.


In [3]:

# ============================================================
# 3.2 HISTORICAL FREQUENCY REFERENCE AUDIT
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. EXISTING BASELINE COUNT METRICS
# ------------------------------------------------------------

BASELINE_METRICS = {
    "email": {
        "days": "email_active_days",
        "p50": "email_count_p50",
        "p75": "email_count_p75",
        "p90": "email_count_p90",
        "p95": "email_count_p95",
        "mad": "email_count_mad"
    },
    "usb": {
        "days": "usb_active_days",
        "p50": "usb_copy_p50",
        "p75": "usb_copy_p75",
        "p90": "usb_copy_p90",
        "p95": "usb_copy_p95",
        "mad": "usb_copy_mad"
    },
    "http": {
        "days": "http_upload_active_days",
        "p50": "http_upload_p50",
        "p75": "http_upload_p75",
        "p90": "http_upload_p90",
        "p95": "http_upload_p95",
        "mad": "http_upload_mad"
    }
}

baseline_columns = [
    "user",
    "baseline_period_start",
    "baseline_period_end",
    "baseline_workdays"
]

for metrics in BASELINE_METRICS.values():
    baseline_columns.extend(metrics.values())

baseline_columns = list(dict.fromkeys(baseline_columns))

# ------------------------------------------------------------
# 2. READ BASELINE — EXISTING VALUES ONLY
# ------------------------------------------------------------

baseline_raw = pd.read_parquet(
    BASELINE_PATH,
    columns=baseline_columns
)

assert baseline_raw["user"].is_unique

candidate_baseline = (
    baseline_raw.loc[
        baseline_raw["user"].isin(candidate_users)
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(candidate_baseline) == 995
assert set(candidate_baseline["user"]) == candidate_users
assert candidate_baseline["baseline_workdays"].gt(0).all()

# Confirm historical period boundaries.
start = pd.to_datetime(
    candidate_baseline["baseline_period_start"]
)
end = pd.to_datetime(
    candidate_baseline["baseline_period_end"]
)

assert start.notna().all()
assert end.notna().all()
assert (start <= end).all()
assert (end < pd.Timestamp("2011-01-01")).all()

# ------------------------------------------------------------
# 3. REUSE NOTEBOOK 01 BASELINE FLAGS
# ------------------------------------------------------------

flag_columns = [
    "user",
    "has_email_channel_baseline",
    "has_usb_channel_baseline",
    "has_http_channel_baseline"
]

existing_flags = pd.read_parquet(
    PROFILE_PATH,
    columns=flag_columns
)

candidate_baseline = candidate_baseline.merge(
    existing_flags,
    on="user",
    how="left",
    validate="one_to_one"
)

assert len(candidate_baseline) == 995

# ------------------------------------------------------------
# 4. CHANNEL-LEVEL REFERENCE STATUS
# ------------------------------------------------------------

reference_parts = []
summary_rows = []

profile_by_user = candidate_profile.set_index("user")

for channel, metrics in BASELINE_METRICS.items():

    historical_days = candidate_baseline[metrics["days"]]

    p50 = candidate_baseline[metrics["p50"]]
    p75 = candidate_baseline[metrics["p75"]]
    p90 = candidate_baseline[metrics["p90"]]
    p95 = candidate_baseline[metrics["p95"]]
    mad = candidate_baseline[metrics["mad"]]

    has_history = historical_days.gt(0)

    # Same availability definition used in Notebook 01.
    has_reference = (
        has_history
        & p50.gt(0)
        & p95.gt(0)
        & p95.ge(p50)
        & mad.notna()
    ).fillna(False)

    # Verify existing Notebook 01 flags — no new classification.
    original_flag = candidate_baseline[
        f"has_{channel}_channel_baseline"
    ].fillna(False).astype(bool)

    assert (
        has_reference.to_numpy()
        == original_flag.to_numpy()
    ).all(), f"{channel}: baseline flag mismatch"

    # Audit the ordering of the existing percentiles.
    ordered_percentiles = (
        p50.le(p75)
        & p75.le(p90)
        & p90.le(p95)
    ).fillna(False)

    invalid_order = has_history & ~ordered_percentiles

    assert not invalid_order.any(), (
        f"{channel}: invalid existing percentile ordering"
    )

    assert not (mad.dropna() < 0).any(), (
        f"{channel}: negative historical MAD"
    )

    # Future channel-active days were already calculated in NB01.
    future_days = (
        candidate_baseline["user"]
        .map(profile_by_user[f"{channel}_days"])
        .astype("int64")
    )

    future_active = future_days.gt(0)

    # Descriptive statuses, not eligibility decisions.
    status = pd.Series(
        "AVAILABLE",
        index=candidate_baseline.index
    )

    status.loc[~has_history] = "NO_HISTORICAL_ACTIVITY"
    status.loc[
        has_history & ~has_reference
    ] = "INCOMPLETE_REFERENCE"

    part = pd.DataFrame({
        "user": candidate_baseline["user"],
        "channel": channel.upper(),
        "future_active_days": future_days,
        "baseline_active_days": historical_days,
        "baseline_p50": p50,
        "baseline_p75": p75,
        "baseline_p90": p90,
        "baseline_p95": p95,
        "baseline_mad": mad,
        "has_baseline_reference": has_reference,
        "reference_status": status
    })

    reference_parts.append(part)

    summary_rows.append({
        "channel": channel.upper(),
        "future_active_users": int(future_active.sum()),
        "historical_active_users": int(has_history.sum()),
        "ready_future_users": int(
            (future_active & has_reference).sum()
        ),
        "future_no_history": int(
            (future_active & ~has_history).sum()
        ),
        "future_incomplete_ref": int(
            (
                future_active
                & has_history
                & ~has_reference
            ).sum()
        )
    })

baseline_reference_audit = pd.concat(
    reference_parts,
    ignore_index=True
)

baseline_reference_summary = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# 5. FINAL VALIDATION
# ------------------------------------------------------------

assert len(baseline_reference_audit) == 995 * 3

assert not baseline_reference_audit.duplicated(
    ["user", "channel"]
).any()

assert baseline_reference_audit["user"].nunique() == 995

for _, row in baseline_reference_summary.iterrows():
    assert (
        row["ready_future_users"]
        + row["future_no_history"]
        + row["future_incomplete_ref"]
        == row["future_active_users"]
    )

# ------------------------------------------------------------
# 6. CONCISE OUTPUT
# ------------------------------------------------------------

print("=" * 72)
print("NOTEBOOK 03 — HISTORICAL FREQUENCY REFERENCE AUDIT")
print("=" * 72)

print(f"Candidate users       : {len(candidate_baseline):,}")
print(f"User-channel records  : {len(baseline_reference_audit):,}")

print("\nBASELINE REFERENCE AVAILABILITY")
print("-" * 72)

print(
    baseline_reference_summary.to_string(index=False)
)

print("\nVALIDATION")
print("Historical period     : PASS")
print("Notebook 01 flags     : PASS")
print("Percentile ordering   : PASS")
print("User-channel coverage : PASS")
print("Baseline recalculated : NO")
print("Source files modified : NO")
print("Files written         : NONE")


NOTEBOOK 03 — HISTORICAL FREQUENCY REFERENCE AUDIT
Candidate users       : 995
User-channel records  : 2,985

BASELINE REFERENCE AVAILABILITY
------------------------------------------------------------------------
channel  future_active_users  historical_active_users  ready_future_users  future_no_history  future_incomplete_ref
  EMAIL                  995                      995                 995                  0                      0
    USB                  719                      721                 719                  0                      0
   HTTP                  567                      567                 567                  0                      0

VALIDATION
Historical period     : PASS
Notebook 01 flags     : PASS
Percentile ordering   : PASS
User-channel coverage : PASS
Baseline recalculated : NO
Source files modified : NO
Files written         : NONE



## 3.3 Baseline–Future Frequency Comparison

**Research Question:** How do naturally observed Future event counts compare with each candidate's existing historical channel-frequency percentiles?

**Objectives:**
- Compare natural Email, USB, and HTTP event counts with the existing Baseline.
- Compare channel-active Future days only, matching the Baseline percentile definitions.
- Classify daily counts into historical frequency bands: ≤P50, P50–P75, P75–P90, P90–P95, and >P95.
- Validate that each comparison uses the correct user, channel, and historical reference.

**Constraints:**
- Frequency/count metrics only.
- No Baseline recalculation or source modification.
- No attack injection or new stealth thresholds.
- Do not interpret natural percentile exceedances as attacks.

**Expected Output:** A validated daily frequency comparison dataset and a concise summary of historical frequency bands.


In [4]:

# ============================================================
# 3.3 BASELINE–FUTURE FREQUENCY COMPARISON
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PREPARE FUTURE CHANNEL-ACTIVE DAYS
# ------------------------------------------------------------

count_to_channel = {
    "external_email_count": "EMAIL",
    "usb_copy_count": "USB",
    "www_upload_count": "HTTP"
}

future_frequency_long = candidate_daily.melt(
    id_vars=["user", "day"],
    value_vars=list(count_to_channel),
    var_name="count_metric",
    value_name="event_count"
)

future_frequency_long["channel"] = (
    future_frequency_long["count_metric"]
    .map(count_to_channel)
)

# Baseline percentiles describe channel-active days.
# Therefore, Future zero-activity days are not compared.
future_frequency_long = (
    future_frequency_long.loc[
        future_frequency_long["event_count"] > 0,
        ["user", "day", "channel", "event_count"]
    ]
    .copy()
)

assert not future_frequency_long.empty

assert not future_frequency_long.duplicated(
    ["user", "day", "channel"]
).any()

# ------------------------------------------------------------
# 2. ATTACH EXISTING HISTORICAL REFERENCES
# ------------------------------------------------------------

reference_columns = [
    "user",
    "channel",
    "baseline_active_days",
    "baseline_p50",
    "baseline_p75",
    "baseline_p90",
    "baseline_p95",
    "baseline_mad",
    "has_baseline_reference"
]

future_frequency_comparison = (
    future_frequency_long.merge(
        baseline_reference_audit[reference_columns],
        on=["user", "channel"],
        how="left",
        validate="many_to_one",
        indicator=True
    )
)

assert (
    future_frequency_comparison["_merge"] == "both"
).all(), "Missing user-channel references"

future_frequency_comparison = (
    future_frequency_comparison.drop(columns="_merge")
)

assert future_frequency_comparison[
    "has_baseline_reference"
].eq(True).all()

percentile_columns = [
    "baseline_p50",
    "baseline_p75",
    "baseline_p90",
    "baseline_p95"
]

assert future_frequency_comparison[
    percentile_columns
].notna().all().all()

# ------------------------------------------------------------
# 3. CLASSIFY NATURAL FUTURE FREQUENCY
# ------------------------------------------------------------

counts = future_frequency_comparison["event_count"]

p50 = future_frequency_comparison["baseline_p50"]
p75 = future_frequency_comparison["baseline_p75"]
p90 = future_frequency_comparison["baseline_p90"]
p95 = future_frequency_comparison["baseline_p95"]

BAND_ORDER = [
    "AT_OR_BELOW_P50",
    "P50_TO_P75",
    "P75_TO_P90",
    "P90_TO_P95",
    "ABOVE_P95"
]

future_frequency_comparison["frequency_band"] = np.select(
    [
        counts <= p50,
        counts <= p75,
        counts <= p90,
        counts <= p95,
        counts > p95
    ],
    BAND_ORDER,
    default="INVALID"
)

assert not (
    future_frequency_comparison["frequency_band"] == "INVALID"
).any()

# Descriptive comparison only.
# These differences are NOT approved injection capacities.
future_frequency_comparison["difference_from_p50"] = (
    counts - p50
)

future_frequency_comparison["difference_from_p95"] = (
    counts - p95
)

future_frequency_comparison = (
    future_frequency_comparison
    .sort_values(["user", "day", "channel"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. RECONCILE WITH NOTEBOOK 01 CHANNEL-ACTIVE DAYS
# ------------------------------------------------------------

observed_days = (
    future_frequency_comparison
    .groupby(["user", "channel"])
    .size()
    .rename("observed_future_days")
    .reset_index()
)

day_reconciliation = baseline_reference_audit[
    ["user", "channel", "future_active_days"]
].merge(
    observed_days,
    on=["user", "channel"],
    how="left",
    validate="one_to_one"
)

day_reconciliation["observed_future_days"] = (
    day_reconciliation["observed_future_days"]
    .fillna(0)
    .astype("int64")
)

assert (
    day_reconciliation["future_active_days"].to_numpy()
    == day_reconciliation["observed_future_days"].to_numpy()
).all(), "Mismatch with Notebook 01 channel-active days"

# ------------------------------------------------------------
# 5. CHANNEL-LEVEL FREQUENCY BAND SUMMARY
# ------------------------------------------------------------

band_counts = pd.crosstab(
    future_frequency_comparison["channel"],
    future_frequency_comparison["frequency_band"]
).reindex(
    index=["EMAIL", "USB", "HTTP"],
    columns=BAND_ORDER,
    fill_value=0
)

channel_coverage = (
    future_frequency_comparison
    .groupby("channel")
    .agg(
        users=("user", "nunique"),
        active_user_days=("day", "size")
    )
    .reindex(["EMAIL", "USB", "HTTP"])
)

frequency_band_summary = pd.concat(
    [channel_coverage, band_counts],
    axis=1
)

assert (
    frequency_band_summary[BAND_ORDER].sum(axis=1)
    == frequency_band_summary["active_user_days"]
).all()

expected_users = (
    baseline_reference_summary.set_index("channel")[
        "ready_future_users"
    ]
)

assert (
    frequency_band_summary["users"]
    == expected_users.reindex(frequency_band_summary.index)
).all()

# ------------------------------------------------------------
# 6. CONCISE OUTPUT
# ------------------------------------------------------------

print("=" * 75)
print("NOTEBOOK 03 — BASELINE–FUTURE FREQUENCY COMPARISON")
print("=" * 75)

print(
    f"Candidates compared  : "
    f"{future_frequency_comparison['user'].nunique():,}"
)

print(
    f"Active channel-days  : "
    f"{len(future_frequency_comparison):,}"
)

print("\nNATURAL FUTURE FREQUENCY BANDS")
print("-" * 75)
print(frequency_band_summary.to_string())

print("\nVALIDATION")
print("Historical references : PASS")
print("User/channel matching : PASS")
print("NB01 active-day counts: PASS")
print("Frequency partition   : PASS")
print("Baseline recalculated : NO")
print("Files written         : NONE")


NOTEBOOK 03 — BASELINE–FUTURE FREQUENCY COMPARISON
Candidates compared  : 995
Active channel-days  : 102,626

NATURAL FUTURE FREQUENCY BANDS
---------------------------------------------------------------------------
         users  active_user_days  AT_OR_BELOW_P50  P50_TO_P75  P75_TO_P90  P90_TO_P95  ABOVE_P95
channel                                                                                         
EMAIL      995             58710            38197       10175        6346        1947       2045
USB        719             30018            18643        6150        3020        1048       1157
HTTP       567             13898             8517        3269        1130         451        531

VALIDATION
Historical references : PASS
User/channel matching : PASS
NB01 active-day counts: PASS
Frequency partition   : PASS
Baseline recalculated : NO
Files written         : NONE



## 3.4 Frequency Deviation Analysis

**Research Question:** How frequently and by how much does each candidate's natural Future activity exceed their existing historical frequency thresholds?

**Objectives:**
- Measure natural P90 and P95 exceedances for each user and channel.
- Calculate the number and percentage of channel-active days above historical P95.
- Quantify the positive difference between observed event counts and historical P95.
- Identify whether natural exceedances are widespread or concentrated among particular candidates.
- Reconcile all results with Section 3.3.

**Constraints:**
- Frequency/event counts only.
- No recalculation or modification of Historical Baseline.
- No injection, user exclusion, or new stealth thresholds.
- Historical percentile exceedances are descriptive, not attack labels or injection budgets.

**Expected Output:** Validated user-channel and user-level natural frequency deviation profiles.


In [5]:

# ============================================================
# 3.4 FREQUENCY DEVIATION ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. USE THE VALIDATED DAILY COMPARISON FROM SECTION 3.3
# ------------------------------------------------------------

deviation_daily = future_frequency_comparison[
    [
        "user",
        "day",
        "channel",
        "event_count",
        "baseline_p90",
        "baseline_p95"
    ]
].copy()

assert not deviation_daily.empty
assert deviation_daily["event_count"].gt(0).all()

# ------------------------------------------------------------
# 2. DESCRIPTIVE DAILY DEVIATIONS
# ------------------------------------------------------------

deviation_daily["above_p90"] = (
    deviation_daily["event_count"]
    > deviation_daily["baseline_p90"]
)

deviation_daily["above_p95"] = (
    deviation_daily["event_count"]
    > deviation_daily["baseline_p95"]
)

# Positive difference between observed counts and P95.
# This is NOT the number of events approved for injection.
deviation_daily["positive_p95_difference"] = (
    deviation_daily["event_count"]
    - deviation_daily["baseline_p95"]
).clip(lower=0)

assert (
    ~deviation_daily["above_p95"]
    | deviation_daily["above_p90"]
).all(), "P95 exceedance must imply P90 exceedance"

assert (
    deviation_daily.loc[
        ~deviation_daily["above_p95"],
        "positive_p95_difference"
    ] == 0
).all()

# ------------------------------------------------------------
# 3. USER-CHANNEL DEVIATION PROFILE
# ------------------------------------------------------------

user_channel_deviation_profile = (
    deviation_daily
    .groupby(["user", "channel"], as_index=False)
    .agg(
        future_active_days=("day", "size"),
        days_above_p90=("above_p90", "sum"),
        days_above_p95=("above_p95", "sum"),
        baseline_p95=("baseline_p95", "first"),
        total_positive_p95_difference=(
            "positive_p95_difference", "sum"
        ),
        max_positive_p95_difference=(
            "positive_p95_difference", "max"
        )
    )
)

user_channel_deviation_profile["p95_exceedance_rate_pct"] = (
    100
    * user_channel_deviation_profile["days_above_p95"]
    / user_channel_deviation_profile["future_active_days"]
)

user_channel_deviation_profile["has_p95_exceedance"] = (
    user_channel_deviation_profile["days_above_p95"] > 0
)

assert (
    user_channel_deviation_profile["days_above_p95"]
    <= user_channel_deviation_profile["days_above_p90"]
).all()

assert (
    user_channel_deviation_profile["days_above_p90"]
    <= user_channel_deviation_profile["future_active_days"]
).all()

assert user_channel_deviation_profile[
    "p95_exceedance_rate_pct"
].between(0, 100).all()

# ------------------------------------------------------------
# 4. USER-LEVEL PROFILE ACROSS NATURALLY ACTIVE CHANNELS
# ------------------------------------------------------------

user_deviation_profile = (
    user_channel_deviation_profile
    .groupby("user", as_index=False)
    .agg(
        observed_channels=("channel", "nunique"),
        active_channel_days=("future_active_days", "sum"),
        days_above_p95=("days_above_p95", "sum"),
        channels_with_p95_exceedance=(
            "has_p95_exceedance", "sum"
        )
    )
)

user_deviation_profile["p95_exceedance_rate_pct"] = (
    100
    * user_deviation_profile["days_above_p95"]
    / user_deviation_profile["active_channel_days"]
)

assert len(user_deviation_profile) == 995
assert user_deviation_profile["user"].is_unique
assert set(user_deviation_profile["user"]) == candidate_users

assert user_deviation_profile["observed_channels"].isin(
    [2, 3]
).all()

# ------------------------------------------------------------
# 5. CHANNEL-LEVEL SUMMARY
# ------------------------------------------------------------

channel_deviation_summary = (
    user_channel_deviation_profile
    .groupby("channel")
    .agg(
        users=("user", "nunique"),
        users_with_exceedance=(
            "has_p95_exceedance", "sum"
        ),
        active_channel_days=("future_active_days", "sum"),
        days_above_p95=("days_above_p95", "sum"),
        median_user_p95_rate_pct=(
            "p95_exceedance_rate_pct", "median"
        ),
        max_positive_p95_difference=(
            "max_positive_p95_difference", "max"
        )
    )
    .reindex(["EMAIL", "USB", "HTTP"])
)

channel_deviation_summary["overall_p95_rate_pct"] = (
    100
    * channel_deviation_summary["days_above_p95"]
    / channel_deviation_summary["active_channel_days"]
)

# ------------------------------------------------------------
# 6. RECONCILE WITH SECTION 3.3
# ------------------------------------------------------------

assert (
    len(user_channel_deviation_profile)
    == int(baseline_reference_summary[
        "ready_future_users"
    ].sum())
)

assert (
    channel_deviation_summary["active_channel_days"]
    .to_numpy()
    ==
    frequency_band_summary["active_user_days"]
    .to_numpy()
).all()

assert (
    channel_deviation_summary["days_above_p95"]
    .to_numpy()
    ==
    frequency_band_summary["ABOVE_P95"]
    .to_numpy()
).all()

assert (
    user_deviation_profile["active_channel_days"].sum()
    == len(future_frequency_comparison)
)

assert (
    user_deviation_profile["days_above_p95"].sum()
    == int(frequency_band_summary["ABOVE_P95"].sum())
)

# ------------------------------------------------------------
# 7. CONCISE OUTPUT
# ------------------------------------------------------------

print("=" * 76)
print("NOTEBOOK 03 — FREQUENCY DEVIATION ANALYSIS")
print("=" * 76)

print(f"Candidates examined   : {len(user_deviation_profile):,}")
print(
    f"Active user-channels  : "
    f"{len(user_channel_deviation_profile):,}"
)
print(
    f"Natural P95 exceedances: "
    f"{int(user_deviation_profile['days_above_p95'].sum()):,}"
)

print("\nCHANNEL-LEVEL DEVIATIONS")
print("-" * 76)

print(
    channel_deviation_summary[
        [
            "users",
            "users_with_exceedance",
            "active_channel_days",
            "days_above_p95",
            "overall_p95_rate_pct",
            "median_user_p95_rate_pct",
            "max_positive_p95_difference"
        ]
    ].round(2).to_string()
)

print("\nCANDIDATE-LEVEL COVERAGE")
print("-" * 76)

print(
    "Users with any natural P95 exceedance :",
    f"{int((user_deviation_profile['days_above_p95'] > 0).sum()):,}"
)

print(
    "Users without natural P95 exceedance  :",
    f"{int((user_deviation_profile['days_above_p95'] == 0).sum()):,}"
)

print("\nVALIDATION")
print("User-channel reconciliation : PASS")
print("Section 3.3 reconciliation  : PASS")
print("Daily deviation consistency : PASS")
print("Baseline recalculated       : NO")
print("Files written               : NONE")


NOTEBOOK 03 — FREQUENCY DEVIATION ANALYSIS
Candidates examined   : 995
Active user-channels  : 2,281
Natural P95 exceedances: 3,733

CHANNEL-LEVEL DEVIATIONS
----------------------------------------------------------------------------
         users  users_with_exceedance  active_channel_days  days_above_p95  overall_p95_rate_pct  median_user_p95_rate_pct  max_positive_p95_difference
channel                                                                                                                                                
EMAIL      995                    660                58710            2045                  3.48                      2.41                          5.8
USB        719                    451                30018            1157                  3.85                      2.86                          6.4
HTTP       567                    284                13898             531                  3.82                      2.56                          6.0

CAND


## 3.5 Frequency Variability & Baseline Consistency

**Research Question:** How closely do candidates' natural Future event counts follow their existing historical frequency patterns?

**Objectives:**
- Measure the absolute and signed differences between Future event counts and historical P50.
- Compare the median absolute Future difference with the existing historical MAD.
- Identify user-channel pairs with zero historical MAD and assess whether Future counts differ from their historical median.
- Distinguish upward, downward, and unchanged typical frequency behavior.
- Validate all results against Sections 3.3 and 3.4.

**Constraints:**
- Event-count frequency only.
- Historical P50 and MAD are read-only reference values.
- Channel-active Future days only.
- No new stealth thresholds, injection budgets, or candidate exclusions.

**Expected Output:** A validated user-channel consistency profile showing how natural Future frequency differs from historical behavior.


In [6]:

# ============================================================
# 3.5 FREQUENCY VARIABILITY & BASELINE CONSISTENCY
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. USE EXISTING DAILY COMPARISON
# ------------------------------------------------------------

consistency_daily = future_frequency_comparison[
    [
        "user",
        "day",
        "channel",
        "event_count",
        "baseline_p50",
        "baseline_mad"
    ]
].copy()

assert consistency_daily["event_count"].gt(0).all()

assert consistency_daily[
    ["baseline_p50", "baseline_mad"]
].notna().all().all()

assert consistency_daily["baseline_mad"].ge(0).all()

# Historical references must remain constant within
# each user-channel pair.

reference_check = (
    consistency_daily
    .groupby(["user", "channel"])
    .agg(
        p50_values=("baseline_p50", "nunique"),
        mad_values=("baseline_mad", "nunique")
    )
)

assert reference_check.eq(1).all().all()

# ------------------------------------------------------------
# 2. BASELINE-RELATIVE DAILY DIFFERENCES
# ------------------------------------------------------------

consistency_daily["signed_p50_gap"] = (
    consistency_daily["event_count"]
    - consistency_daily["baseline_p50"]
)

consistency_daily["absolute_p50_gap"] = (
    consistency_daily["signed_p50_gap"].abs()
)

consistency_daily["differs_from_baseline_p50"] = (
    ~np.isclose(
        consistency_daily["signed_p50_gap"],
        0,
        atol=1e-9,
        rtol=0
    )
)

# ------------------------------------------------------------
# 3. USER-CHANNEL CONSISTENCY PROFILE
# ------------------------------------------------------------

user_channel_consistency = (
    consistency_daily
    .groupby(["user", "channel"], as_index=False)
    .agg(
        future_active_days=("day", "size"),
        baseline_p50=("baseline_p50", "first"),
        baseline_mad=("baseline_mad", "first"),
        median_signed_p50_gap=(
            "signed_p50_gap", "median"
        ),
        median_absolute_p50_gap=(
            "absolute_p50_gap", "median"
        ),
        max_absolute_p50_gap=(
            "absolute_p50_gap", "max"
        ),
        days_different_from_p50=(
            "differs_from_baseline_p50", "sum"
        )
    )
)

# ------------------------------------------------------------
# 4. COMPARE FUTURE DIFFERENCE WITH HISTORICAL MAD
# ------------------------------------------------------------

user_channel_consistency["historical_mad_zero"] = (
    user_channel_consistency["baseline_mad"].eq(0)
)

positive_mad = (
    user_channel_consistency["baseline_mad"].gt(0)
)

# Undefined when historical MAD = 0.
# Never substitute an arbitrary denominator.

user_channel_consistency["relative_median_gap_mad"] = (
    user_channel_consistency["median_absolute_p50_gap"]
    .div(
        user_channel_consistency["baseline_mad"].where(
            positive_mad
        )
    )
)

user_channel_consistency[
    "median_gap_exceeds_historical_mad"
] = (
    positive_mad
    & (
        user_channel_consistency[
            "median_absolute_p50_gap"
        ]
        > user_channel_consistency["baseline_mad"]
    )
)

user_channel_consistency[
    "zero_mad_with_future_difference"
] = (
    user_channel_consistency["historical_mad_zero"]
    & user_channel_consistency[
        "days_different_from_p50"
    ].gt(0)
)

# Descriptive direction of the typical Future frequency.
# No arbitrary cutoff is introduced.

signed_gap = (
    user_channel_consistency["median_signed_p50_gap"]
)

user_channel_consistency["frequency_direction"] = np.select(
    [
        signed_gap > 1e-9,
        signed_gap < -1e-9
    ],
    [
        "ABOVE_BASELINE_MEDIAN",
        "BELOW_BASELINE_MEDIAN"
    ],
    default="EQUAL_BASELINE_MEDIAN"
)

# ------------------------------------------------------------
# 5. CHANNEL-LEVEL SUMMARY
# ------------------------------------------------------------

consistency_summary = (
    user_channel_consistency
    .groupby("channel")
    .agg(
        user_channels=("user", "size"),
        positive_mad_users=(
            "baseline_mad",
            lambda x: int(x.gt(0).sum())
        ),
        zero_mad_users=(
            "historical_mad_zero", "sum"
        ),
        median_absolute_gap=(
            "median_absolute_p50_gap", "median"
        ),
        median_relative_gap_mad=(
            "relative_median_gap_mad", "median"
        ),
        users_gap_above_historical_mad=(
            "median_gap_exceeds_historical_mad", "sum"
        ),
        zero_mad_users_with_future_difference=(
            "zero_mad_with_future_difference", "sum"
        )
    )
    .reindex(["EMAIL", "USB", "HTTP"])
)

direction_summary = pd.crosstab(
    user_channel_consistency["channel"],
    user_channel_consistency["frequency_direction"]
).reindex(
    index=["EMAIL", "USB", "HTTP"],
    columns=[
        "ABOVE_BASELINE_MEDIAN",
        "BELOW_BASELINE_MEDIAN",
        "EQUAL_BASELINE_MEDIAN"
    ],
    fill_value=0
)

# ------------------------------------------------------------
# 6. RECONCILIATION
# ------------------------------------------------------------

assert len(user_channel_consistency) == 2281

assert user_channel_consistency["user"].nunique() == 995

assert not user_channel_consistency.duplicated(
    ["user", "channel"]
).any()

assert (
    user_channel_consistency["future_active_days"].sum()
    == len(future_frequency_comparison)
)

assert (
    consistency_summary["user_channels"].to_numpy()
    ==
    channel_deviation_summary["users"].to_numpy()
).all()

assert (
    consistency_summary["positive_mad_users"]
    + consistency_summary["zero_mad_users"]
    == consistency_summary["user_channels"]
).all()

assert (
    direction_summary.sum(axis=1).to_numpy()
    == consistency_summary["user_channels"].to_numpy()
).all()

assert user_channel_consistency.loc[
    user_channel_consistency["historical_mad_zero"],
    "relative_median_gap_mad"
].isna().all()

assert user_channel_consistency.loc[
    positive_mad,
    "relative_median_gap_mad"
].notna().all()

# ------------------------------------------------------------
# 7. CONCISE OUTPUT
# ------------------------------------------------------------

print("=" * 76)
print("NOTEBOOK 03 — FREQUENCY VARIABILITY & BASELINE CONSISTENCY")
print("=" * 76)

print(
    f"Candidates examined  : "
    f"{user_channel_consistency['user'].nunique():,}"
)

print(
    f"Active user-channels : "
    f"{len(user_channel_consistency):,}"
)

print("\nHISTORICAL MAD & FUTURE CONSISTENCY")
print("-" * 76)
print(consistency_summary.round(3).to_string())

print("\nTYPICAL FUTURE FREQUENCY DIRECTION")
print("-" * 76)
print(direction_summary.to_string())

print("\nVALIDATION")
print("Reference consistency : PASS")
print("User-channel coverage : PASS")
print("Section 3.4 alignment : PASS")
print("Zero-MAD handling     : PASS")
print("Baseline recalculated : NO")
print("Files written         : NONE")


NOTEBOOK 03 — FREQUENCY VARIABILITY & BASELINE CONSISTENCY
Candidates examined  : 995
Active user-channels : 2,281

HISTORICAL MAD & FUTURE CONSISTENCY
----------------------------------------------------------------------------
         user_channels  positive_mad_users  zero_mad_users  median_absolute_gap  median_relative_gap_mad  users_gap_above_historical_mad  zero_mad_users_with_future_difference
channel                                                                                                                                                                        
EMAIL              995                 594             401                  1.0                      1.0                               8                                    213
USB                719                 290             429                  0.0                      1.0                              38                                    401
HTTP               567                 296             271         


## 3.6 Temporal Frequency Stability

**Research Question:** How does natural event frequency change across the Future months for each candidate and channel?

**Objectives:**
- Analyze monthly event-count frequency for the 995 candidates.
- Calculate events per eligible workday and per channel-active day.
- Compare monthly frequency patterns without repeating Notebook 01's channel-activity audit.
- Measure within-user monthly frequency variation.
- Exclude partial June 2011 from the primary month-to-month comparison while retaining it for reconciliation.

**Constraints:**
- Frequency and event counts only.
- No Baseline recalculation or modification.
- No attack injection or new eligibility thresholds.
- No automatic stable/unstable classification.

**Expected Output:** Monthly frequency profiles and a validated user-channel temporal variability summary.


In [7]:

# ============================================================
# 3.6 TEMPORAL FREQUENCY STABILITY
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PREPARE MONTHLY NATURAL EVENT COUNTS
# ------------------------------------------------------------

monthly_source = candidate_daily[
    ["user", "day", *CHANNELS.values()]
].copy()

monthly_source["month"] = (
    monthly_source["day"].dt.to_period("M")
)

count_to_channel = {
    "external_email_count": "EMAIL",
    "usb_copy_count": "USB",
    "www_upload_count": "HTTP"
}

monthly_daily_long = monthly_source.melt(
    id_vars=["user", "day", "month"],
    value_vars=list(count_to_channel),
    var_name="count_metric",
    value_name="event_count"
)

monthly_daily_long["channel"] = (
    monthly_daily_long["count_metric"]
    .map(count_to_channel)
)

# ------------------------------------------------------------
# 2. MONTHLY USER-CHANNEL FREQUENCY
# ------------------------------------------------------------

monthly_frequency = (
    monthly_daily_long
    .groupby(["user", "channel", "month"], as_index=False)
    .agg(
        eligible_workdays=("day", "size"),
        active_days=(
            "event_count",
            lambda s: int(s.gt(0).sum())
        ),
        total_events=("event_count", "sum")
    )
)

# Keep only user-channel pairs that were naturally
# active during Future, as confirmed in Section 3.2.

active_pairs = baseline_reference_audit.loc[
    baseline_reference_audit["future_active_days"] > 0,
    ["user", "channel"]
]

monthly_frequency = monthly_frequency.merge(
    active_pairs,
    on=["user", "channel"],
    how="inner",
    validate="many_to_one"
)

monthly_frequency["events_per_eligible_day"] = (
    monthly_frequency["total_events"]
    / monthly_frequency["eligible_workdays"]
)

monthly_frequency["events_per_active_day"] = (
    monthly_frequency["total_events"]
    / monthly_frequency["active_days"].replace(0, np.nan)
)

assert not monthly_frequency.duplicated(
    ["user", "channel", "month"]
).any()

# ------------------------------------------------------------
# 3. FULL-MONTH COMPARISON — JANUARY TO MAY
# ------------------------------------------------------------

FULL_MONTHS = pd.period_range(
    "2011-01", "2011-05", freq="M"
)

monthly_full = monthly_frequency.loc[
    monthly_frequency["month"].isin(FULL_MONTHS)
].copy()

# June 2011 is retained in monthly_frequency, but excluded
# from month-to-month stability calculations.

assert not monthly_full.empty

assert monthly_full["month"].isin(FULL_MONTHS).all()

# ------------------------------------------------------------
# 4. PER-USER, PER-CHANNEL MONTHLY VARIATION
# ------------------------------------------------------------

temporal_frequency_profile = (
    monthly_full
    .groupby(["user", "channel"], as_index=False)
    .agg(
        months_observed=("month", "nunique"),
        months_with_activity=(
            "active_days",
            lambda s: int(s.gt(0).sum())
        ),
        min_events_per_eligible_day=(
            "events_per_eligible_day", "min"
        ),
        max_events_per_eligible_day=(
            "events_per_eligible_day", "max"
        ),
        min_events_per_active_day=(
            "events_per_active_day", "min"
        ),
        max_events_per_active_day=(
            "events_per_active_day", "max"
        )
    )
)

# Monthly frequency range is descriptive.
# No stability threshold is imposed.

temporal_frequency_profile[
    "eligible_day_frequency_range"
] = (
    temporal_frequency_profile["max_events_per_eligible_day"]
    - temporal_frequency_profile["min_events_per_eligible_day"]
).where(
    temporal_frequency_profile["months_observed"] >= 2
)

temporal_frequency_profile[
    "active_day_frequency_range"
] = (
    temporal_frequency_profile["max_events_per_active_day"]
    - temporal_frequency_profile["min_events_per_active_day"]
).where(
    temporal_frequency_profile["months_with_activity"] >= 2
)

# ------------------------------------------------------------
# 5. MONTHLY CHANNEL-LEVEL FREQUENCY
# ------------------------------------------------------------

monthly_channel_summary = (
    monthly_full
    .groupby(["month", "channel"], as_index=False)
    .agg(
        total_events=("total_events", "sum"),
        eligible_workdays=("eligible_workdays", "sum"),
        active_channel_days=("active_days", "sum")
    )
)

monthly_channel_summary["events_per_eligible_day"] = (
    monthly_channel_summary["total_events"]
    / monthly_channel_summary["eligible_workdays"]
)

monthly_channel_summary["events_per_active_day"] = (
    monthly_channel_summary["total_events"]
    / monthly_channel_summary[
        "active_channel_days"
    ].replace(0, np.nan)
)

variability_summary = (
    temporal_frequency_profile
    .groupby("channel")
    .agg(
        user_channels=("user", "size"),
        users_with_2plus_active_months=(
            "months_with_activity",
            lambda s: int(s.ge(2).sum())
        ),
        median_eligible_day_range=(
            "eligible_day_frequency_range", "median"
        ),
        median_active_day_range=(
            "active_day_frequency_range", "median"
        )
    )
    .reindex(["EMAIL", "USB", "HTTP"])
)

# ------------------------------------------------------------
# 6. RECONCILIATION — INCLUDING PARTIAL JUNE
# ------------------------------------------------------------

# Monthly event totals must equal the original
# candidate Future event totals.

for metric, channel in count_to_channel.items():

    original_total = candidate_daily[metric].sum()

    monthly_total = monthly_frequency.loc[
        monthly_frequency["channel"] == channel,
        "total_events"
    ].sum()

    assert np.isclose(
        original_total, monthly_total
    ), f"{channel}: event-count mismatch"

# Every channel-active Future day must be represented.

monthly_active_totals = (
    monthly_frequency.groupby("channel")["active_days"]
    .sum()
    .reindex(["EMAIL", "USB", "HTTP"])
)

expected_active_totals = (
    frequency_band_summary["active_user_days"]
)

assert (
    monthly_active_totals.to_numpy()
    == expected_active_totals.to_numpy()
).all()

assert (
    temporal_frequency_profile["user"].nunique() == 995
)

assert not temporal_frequency_profile.duplicated(
    ["user", "channel"]
).any()

assert (
    temporal_frequency_profile["months_with_activity"]
    <= temporal_frequency_profile["months_observed"]
).all()

assert (
    temporal_frequency_profile[
        "eligible_day_frequency_range"
    ].dropna() >= 0
).all()

assert (
    temporal_frequency_profile[
        "active_day_frequency_range"
    ].dropna() >= 0
).all()

# ------------------------------------------------------------
# 7. CONCISE OUTPUT
# ------------------------------------------------------------

print("=" * 76)
print("NOTEBOOK 03 — TEMPORAL FREQUENCY STABILITY")
print("=" * 76)

print(f"Candidates examined     : {temporal_frequency_profile['user'].nunique():,}")
print(f"Active user-channels    : {len(temporal_frequency_profile):,}")
print("Primary months          : January–May 2011")
print("Partial June            : Retained; excluded from stability comparison")

print("\nMONTHLY EVENTS PER CHANNEL-ACTIVE DAY")
print("-" * 76)

print(
    monthly_channel_summary.pivot(
        index="month",
        columns="channel",
        values="events_per_active_day"
    )
    .reindex(columns=["EMAIL", "USB", "HTTP"])
    .round(3)
    .to_string()
)

print("\nUSER-CHANNEL MONTHLY VARIABILITY")
print("-" * 76)
print(variability_summary.round(3).to_string())

print("\nVALIDATION")
print("Future event totals       : PASS")
print("Channel-active day totals : PASS")
print("User-channel uniqueness   : PASS")
print("Partial-month handling    : PASS")
print("Baseline recalculated     : NO")
print("Files written             : NONE")


NOTEBOOK 03 — TEMPORAL FREQUENCY STABILITY
Candidates examined     : 995
Active user-channels    : 2,281
Primary months          : January–May 2011
Partial June            : Retained; excluded from stability comparison

MONTHLY EVENTS PER CHANNEL-ACTIVE DAY
----------------------------------------------------------------------------
channel  EMAIL    USB   HTTP
month                       
2011-01  2.218  2.957  3.601
2011-02  2.119  2.982  3.496
2011-03  2.119  2.948  3.469
2011-04  2.260  3.010  3.520
2011-05  2.202  2.998  3.474

USER-CHANNEL MONTHLY VARIABILITY
----------------------------------------------------------------------------
         user_channels  users_with_2plus_active_months  median_eligible_day_range  median_active_day_range
channel                                                                                                   
EMAIL              995                             990                      0.609                    0.630
USB                719        


## 3.7 Cross-Channel Frequency Deviation

**Research Question:** Do naturally co-active channels show simultaneous event-frequency deviations from their existing historical baselines?

**Objectives:**
- Compare user-specific P95 exceedance rates across naturally active channels.
- Identify eligible Future days where two or three channels are naturally active.
- Measure whether historical P95 exceedances occur simultaneously across channels.
- Examine Email–USB, Email–HTTP, and USB–HTTP pairs.
- Reconcile all findings with the earlier frequency audits.

**Constraints:**
- Event counts/frequency only.
- Reuse existing daily comparisons; do not repeat the temporal rotation audit.
- Read-only Baseline; no recalculation or modification.
- No injection, scenario assignment, or new thresholds.

**Expected Output:** User-level cross-channel deviation profiles, same-day deviation summaries, and pairwise frequency-deviation statistics.


In [8]:

# ============================================================
# 3.7 CROSS-CHANNEL FREQUENCY DEVIATION
# ============================================================

import numpy as np
import pandas as pd

CHANNEL_ORDER = ["EMAIL", "USB", "HTTP"]

# ------------------------------------------------------------
# 1. USER-LEVEL CROSS-CHANNEL DEVIATION PROFILE
# ------------------------------------------------------------

user_cross_channel_profile = (
    user_channel_deviation_profile
    .groupby("user", as_index=False)
    .agg(
        naturally_active_channels=("channel", "nunique"),
        min_p95_rate_pct=("p95_exceedance_rate_pct", "min"),
        max_p95_rate_pct=("p95_exceedance_rate_pct", "max"),
        channels_with_any_p95_exceedance=(
            "has_p95_exceedance", "sum"
        )
    )
)

user_cross_channel_profile["p95_rate_range_pct"] = (
    user_cross_channel_profile["max_p95_rate_pct"]
    - user_cross_channel_profile["min_p95_rate_pct"]
)

assert len(user_cross_channel_profile) == 995

assert user_cross_channel_profile[
    "naturally_active_channels"
].isin([2, 3]).all()

assert user_cross_channel_profile[
    "p95_rate_range_pct"
].between(0, 100).all()

# ------------------------------------------------------------
# 2. SAME-DAY CHANNEL ACTIVITY AND P95 EXCEEDANCES
# ------------------------------------------------------------

# The source includes channel-active Future days only.
# Missing channel values after pivot mean the channel
# was naturally inactive on that particular date.

daily_channel_p95 = deviation_daily.pivot(
    index=["user", "day"],
    columns="channel",
    values="above_p95"
).reindex(columns=CHANNEL_ORDER)

naturally_active = daily_channel_p95.notna()
above_historical_p95 = daily_channel_p95.eq(True)

# Reconcile daily channel counts with the original
# eligible Future user-day dataset.

expected_active_channels = (
    candidate_daily
    .set_index(["user", "day"])[
        list(CHANNELS.values())
    ]
    .gt(0)
    .sum(axis=1)
)

observed_active_channels = (
    naturally_active.sum(axis=1)
    .reindex(
        expected_active_channels.index,
        fill_value=0
    )
)

assert np.array_equal(
    expected_active_channels.to_numpy(),
    observed_active_channels.to_numpy()
)

# ------------------------------------------------------------
# 3. DAYS WITH MULTIPLE NATURALLY ACTIVE CHANNELS
# ------------------------------------------------------------

multi_channel_frequency_days = pd.DataFrame({
    "naturally_active_channels": naturally_active.sum(axis=1),
    "channels_above_p95": above_historical_p95.sum(axis=1)
})

multi_channel_frequency_days = (
    multi_channel_frequency_days.loc[
        multi_channel_frequency_days[
            "naturally_active_channels"
        ] >= 2
    ]
    .reset_index()
)

assert not multi_channel_frequency_days.empty

assert (
    multi_channel_frequency_days["channels_above_p95"]
    <= multi_channel_frequency_days[
        "naturally_active_channels"
    ]
).all()

same_day_deviation_summary = pd.DataFrame({
    "metric": [
        "Multi-channel user-days",
        "Days with exactly 2 active channels",
        "Days with exactly 3 active channels",
        "Days with no P95 exceedance",
        "Days with exactly 1 P95 exceedance",
        "Days with 2+ simultaneous P95 exceedances",
        "Days with all active channels above P95"
    ],
    "value": [
        len(multi_channel_frequency_days),
        int((
            multi_channel_frequency_days[
                "naturally_active_channels"
            ] == 2
        ).sum()),
        int((
            multi_channel_frequency_days[
                "naturally_active_channels"
            ] == 3
        ).sum()),
        int((
            multi_channel_frequency_days[
                "channels_above_p95"
            ] == 0
        ).sum()),
        int((
            multi_channel_frequency_days[
                "channels_above_p95"
            ] == 1
        ).sum()),
        int((
            multi_channel_frequency_days[
                "channels_above_p95"
            ] >= 2
        ).sum()),
        int((
            multi_channel_frequency_days[
                "channels_above_p95"
            ]
            ==
            multi_channel_frequency_days[
                "naturally_active_channels"
            ]
        ).sum())
    ]
})

# ------------------------------------------------------------
# 4. PAIRWISE SIMULTANEOUS DEVIATIONS
# ------------------------------------------------------------

CHANNEL_PAIRS = [
    ("EMAIL", "USB"),
    ("EMAIL", "HTTP"),
    ("USB", "HTTP")
]

pair_results = []

for channel_a, channel_b in CHANNEL_PAIRS:

    # A pair is considered only if both channels were
    # naturally active on the same eligible user-day.
    both_active = (
        naturally_active[channel_a]
        & naturally_active[channel_b]
    )

    exceed_a = above_historical_p95[channel_a]
    exceed_b = above_historical_p95[channel_b]

    both_exceed = both_active & exceed_a & exceed_b

    exactly_one_exceeds = (
        both_active
        & (exceed_a ^ exceed_b)
    )

    neither_exceeds = (
        both_active
        & ~exceed_a
        & ~exceed_b
    )

    shared_days = int(both_active.sum())

    pair_results.append({
        "channel_pair": f"{channel_a}-{channel_b}",
        "users_with_pair_days": (
            daily_channel_p95.index
            .get_level_values("user")[both_active]
            .nunique()
        ),
        "both_active_days": shared_days,
        "both_above_p95_days": int(both_exceed.sum()),
        "exactly_one_above_p95_days": int(
            exactly_one_exceeds.sum()
        ),
        "neither_above_p95_days": int(
            neither_exceeds.sum()
        ),
        "both_above_p95_rate_pct": (
            100 * both_exceed.sum() / shared_days
            if shared_days else np.nan
        )
    })

pair_deviation_summary = pd.DataFrame(pair_results)

# ------------------------------------------------------------
# 5. RECONCILIATION
# ------------------------------------------------------------

assert (
    pair_deviation_summary["both_above_p95_days"]
    + pair_deviation_summary[
        "exactly_one_above_p95_days"
    ]
    + pair_deviation_summary["neither_above_p95_days"]
    ==
    pair_deviation_summary["both_active_days"]
).all()

assert (
    same_day_deviation_summary.loc[
        same_day_deviation_summary["metric"].isin([
            "Days with no P95 exceedance",
            "Days with exactly 1 P95 exceedance",
            "Days with 2+ simultaneous P95 exceedances"
        ]),
        "value"
    ].sum()
    == len(multi_channel_frequency_days)
)

for channel in CHANNEL_ORDER:

    assert int(naturally_active[channel].sum()) == int(
        frequency_band_summary.loc[
            channel, "active_user_days"
        ]
    )

    assert int(above_historical_p95[channel].sum()) == int(
        frequency_band_summary.loc[
            channel, "ABOVE_P95"
        ]
    )

assert (
    user_cross_channel_profile["user"].nunique() == 995
)

# ------------------------------------------------------------
# 6. CONCISE OUTPUT
# ------------------------------------------------------------

print("=" * 76)
print("NOTEBOOK 03 — CROSS-CHANNEL FREQUENCY DEVIATION")
print("=" * 76)

print(f"Candidates examined       : {len(user_cross_channel_profile):,}")
print(f"Multi-channel user-days   : {len(multi_channel_frequency_days):,}")

print("\nSAME-DAY P95 DEVIATION")
print("-" * 76)
print(same_day_deviation_summary.to_string(index=False))

print("\nPAIRWISE DEVIATION")
print("-" * 76)
print(pair_deviation_summary.round(2).to_string(index=False))

print("\nUSER-LEVEL CROSS-CHANNEL COMPARISON")
print("-" * 76)

print(
    "Median P95-rate range across channels (%):",
    round(
        user_cross_channel_profile[
            "p95_rate_range_pct"
        ].median(),
        3
    )
)

print(
    "Users with P95 exceedances in 2+ channels:",
    int((
        user_cross_channel_profile[
            "channels_with_any_p95_exceedance"
        ] >= 2
    ).sum())
)

print("\nVALIDATION")
print("Daily channel reconciliation : PASS")
print("Historical P95 alignment     : PASS")
print("Pairwise partition           : PASS")
print("User coverage                : PASS")
print("Baseline recalculated        : NO")
print("Files written                : NONE")


NOTEBOOK 03 — CROSS-CHANNEL FREQUENCY DEVIATION
Candidates examined       : 995
Multi-channel user-days   : 24,625

SAME-DAY P95 DEVIATION
----------------------------------------------------------------------------
                                   metric  value
                  Multi-channel user-days  24625
      Days with exactly 2 active channels  23215
      Days with exactly 3 active channels   1410
              Days with no P95 exceedance  22802
       Days with exactly 1 P95 exceedance   1772
Days with 2+ simultaneous P95 exceedances     51
  Days with all active channels above P95     43

PAIRWISE DEVIATION
----------------------------------------------------------------------------
channel_pair  users_with_pair_days  both_active_days  both_above_p95_days  exactly_one_above_p95_days  neither_above_p95_days  both_above_p95_rate_pct
   EMAIL-USB                   700             17198                   34                        1196                   15968                   


## 3.8 Candidate Behavioral Assessment

**Research Question:** What is the consolidated frequency-based behavioral profile of each of the 995 multi-channel candidates?

**Objectives:**
- Combine previously validated Baseline references, Future frequency deviations, and temporal variability.
- Build a user-channel profile for every naturally active channel.
- Build a user-level profile covering all 995 candidates.
- Summarize individual and simultaneous P95 exceedances without repeating earlier analyses.
- Preserve the distinction between natural behavior and actual injection feasibility.

**Constraints:**
- Frequency/event counts only.
- No Baseline recalculation or source modification.
- No candidate exclusion, attack labeling, or scenario assignment.
- Reuse validated outputs from Sections 3.1–3.7.

**Expected Output:** Two consolidated, validated behavioral profiles held in memory for final quality checks and export.


In [10]:

# ============================================================
# 3.8 CANDIDATE BEHAVIORAL ASSESSMENT
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. EXISTING USER-CHANNEL BASELINE REFERENCES
# ------------------------------------------------------------

reference_columns = [
    "user", "channel",
    "future_active_days", "baseline_active_days",
    "baseline_p50", "baseline_p75",
    "baseline_p90", "baseline_p95",
    "baseline_mad", "has_baseline_reference"
]

active_references = baseline_reference_audit.loc[
    baseline_reference_audit["future_active_days"] > 0,
    reference_columns
].copy()

assert len(active_references) == 2281
assert active_references["has_baseline_reference"].all()

# ------------------------------------------------------------
# 2. COMBINE PREVIOUSLY VALIDATED CHANNEL-LEVEL RESULTS
# ------------------------------------------------------------

deviation_columns = [
    "user", "channel",
    "days_above_p90",
    "days_above_p95",
    "p95_exceedance_rate_pct",
    "has_p95_exceedance",
    "total_positive_p95_difference",
    "max_positive_p95_difference"
]

consistency_columns = [
    "user", "channel",
    "median_signed_p50_gap",
    "median_absolute_p50_gap",
    "historical_mad_zero",
    "relative_median_gap_mad",
    "median_gap_exceeds_historical_mad",
    "zero_mad_with_future_difference",
    "frequency_direction"
]

temporal_columns = [
    "user", "channel",
    "months_observed",
    "months_with_activity",
    "eligible_day_frequency_range",
    "active_day_frequency_range"
]

candidate_channel_behavior_profile = (
    active_references
    .merge(
        user_channel_deviation_profile[deviation_columns],
        on=["user", "channel"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        user_channel_consistency[consistency_columns],
        on=["user", "channel"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        temporal_frequency_profile[temporal_columns],
        on=["user", "channel"],
        how="left",
        validate="one_to_one"
    )
    .sort_values(["user", "channel"])
    .reset_index(drop=True)
)

assert len(candidate_channel_behavior_profile) == 2281

assert not candidate_channel_behavior_profile.duplicated(
    ["user", "channel"]
).any()

assert candidate_channel_behavior_profile[
    ["days_above_p95", "median_absolute_p50_gap",
     "months_observed"]
].notna().all().all()

# Verify that all original active-day counts are preserved.
source_days = (
    user_channel_deviation_profile
    .set_index(["user", "channel"])["future_active_days"]
)

consolidated_days = (
    candidate_channel_behavior_profile
    .set_index(["user", "channel"])["future_active_days"]
)

pd.testing.assert_series_equal(
    consolidated_days.sort_index().astype("int64"),
    source_days.sort_index().astype("int64"),
    check_names=False
)

# ------------------------------------------------------------
# 3. SUMMARIZE EXISTING RESULTS AT USER LEVEL
# ------------------------------------------------------------

user_frequency_rollup = (
    candidate_channel_behavior_profile
    .groupby("user", as_index=False)
    .agg(
        active_channel_days=("future_active_days", "sum"),
        natural_p95_exceedance_channel_days=(
            "days_above_p95", "sum"
        ),
        channels_with_zero_mad=(
            "historical_mad_zero", "sum"
        ),
        channels_with_median_gap_above_mad=(
            "median_gap_exceeds_historical_mad", "sum"
        )
    )
)

# Reuse Section 3.7's validated same-day results.
multi_day_source = multi_channel_frequency_days[
    ["user", "day",
     "naturally_active_channels",
     "channels_above_p95"]
].copy()

multi_day_source["any_p95_exceedance"] = (
    multi_day_source["channels_above_p95"] >= 1
)

multi_day_source["simultaneous_p95_exceedance"] = (
    multi_day_source["channels_above_p95"] >= 2
)

multi_day_source["all_active_above_p95"] = (
    multi_day_source["channels_above_p95"]
    == multi_day_source["naturally_active_channels"]
)

user_same_day_summary = (
    multi_day_source
    .groupby("user", as_index=False)
    .agg(
        multi_channel_user_days=("day", "size"),
        multi_channel_days_with_any_p95=(
            "any_p95_exceedance", "sum"
        ),
        days_with_simultaneous_p95=(
            "simultaneous_p95_exceedance", "sum"
        ),
        days_all_active_above_p95=(
            "all_active_above_p95", "sum"
        )
    )
)

# ------------------------------------------------------------
# 4. BUILD THE FINAL CANDIDATE-LEVEL PROFILE
# ------------------------------------------------------------

candidate_behavior_profile = (
    candidate_profile[
        ["user", "future_workdays",
         "active_channel_count", "observed_channel_set"]
    ]
    .merge(
        user_cross_channel_profile,
        on="user",
        how="left",
        validate="one_to_one"
    )
    .merge(
        user_frequency_rollup,
        on="user",
        how="left",
        validate="one_to_one"
    )
    .merge(
        user_same_day_summary,
        on="user",
        how="left",
        validate="one_to_one"
    )
)

same_day_columns = [
    "multi_channel_user_days",
    "multi_channel_days_with_any_p95",
    "days_with_simultaneous_p95",
    "days_all_active_above_p95"
]

# A candidate may use two channels on different dates
# without ever having a multi-channel day.
candidate_behavior_profile[same_day_columns] = (
    candidate_behavior_profile[same_day_columns]
    .fillna(0)
    .astype("int64")
)

candidate_behavior_profile = (
    candidate_behavior_profile
    .sort_values("user")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. COMPREHENSIVE RECONCILIATION
# ------------------------------------------------------------

assert len(candidate_behavior_profile) == 995
assert candidate_behavior_profile["user"].is_unique
assert set(candidate_behavior_profile["user"]) == candidate_users

assert (
    candidate_behavior_profile["active_channel_count"]
    == candidate_behavior_profile["naturally_active_channels"]
).all()

assert (
    candidate_behavior_profile["future_workdays"].sum()
    == len(candidate_daily)
    == 103395
)

assert (
    candidate_behavior_profile["active_channel_days"].sum()
    == len(future_frequency_comparison)
    == 102626
)

assert (
    candidate_behavior_profile[
        "natural_p95_exceedance_channel_days"
    ].sum()
    == int(frequency_band_summary["ABOVE_P95"].sum())
    == 3733
)

assert (
    candidate_behavior_profile[
        "multi_channel_user_days"
    ].sum()
    == len(multi_channel_frequency_days)
    == 24625
)

assert (
    candidate_behavior_profile[
        "days_with_simultaneous_p95"
    ].sum()
    == 51
)

assert (
    candidate_behavior_profile[
        "days_all_active_above_p95"
    ].sum()
    == 43
)

assert (
    candidate_behavior_profile[
        "days_with_simultaneous_p95"
    ]
    <= candidate_behavior_profile[
        "multi_channel_user_days"
    ]
).all()

assert (
    candidate_behavior_profile[
        "channels_with_zero_mad"
    ]
    <= candidate_behavior_profile[
        "active_channel_count"
    ]
).all()

# ------------------------------------------------------------
# 6. NEW CONSOLIDATED CANDIDATE SUMMARY
# ------------------------------------------------------------

candidate_behavior_profile["has_any_p95_exceedance"] = (
    candidate_behavior_profile[
        "natural_p95_exceedance_channel_days"
    ] > 0
)

candidate_behavior_profile["has_simultaneous_p95"] = (
    candidate_behavior_profile[
        "days_with_simultaneous_p95"
    ] > 0
)

candidate_behavior_profile["has_zero_mad_channel"] = (
    candidate_behavior_profile[
        "channels_with_zero_mad"
    ] > 0
)

candidate_assessment_summary = (
    candidate_behavior_profile
    .groupby("active_channel_count")
    .agg(
        candidates=("user", "size"),
        users_with_any_p95=(
            "has_any_p95_exceedance", "sum"
        ),
        users_with_simultaneous_p95=(
            "has_simultaneous_p95", "sum"
        ),
        users_with_zero_mad=(
            "has_zero_mad_channel", "sum"
        ),
        median_multi_channel_days=(
            "multi_channel_user_days", "median"
        ),
        median_p95_rate_range_pct=(
            "p95_rate_range_pct", "median"
        )
    )
)

assert candidate_assessment_summary[
    "candidates"
].sum() == 995

# ------------------------------------------------------------
# 7. CONCISE OUTPUT
# ------------------------------------------------------------

print("=" * 76)
print("NOTEBOOK 03 — CANDIDATE BEHAVIORAL ASSESSMENT")
print("=" * 76)

print(f"Candidates consolidated   : {len(candidate_behavior_profile):,}")
print(f"Active user-channel pairs : {len(candidate_channel_behavior_profile):,}")
print(f"Eligible Future user-days : {len(candidate_daily):,}")
print(f"Active channel-days       : {len(future_frequency_comparison):,}")

print("\nBEHAVIORAL ASSESSMENT BY CHANNEL DIVERSITY")
print("-" * 76)
print(candidate_assessment_summary.round(3).to_string())

print("\nVALIDATION")
print("Candidate population          : PASS")
print("User-channel reconciliation   : PASS")
print("Future event-day reconciliation: PASS")
print("P95 exceedance reconciliation : PASS")
print("Simultaneous-day reconciliation: PASS")
print("Baseline recalculated         : NO")
print("Users excluded                : NONE")
print("Files written                 : NONE")


NOTEBOOK 03 — CANDIDATE BEHAVIORAL ASSESSMENT
Candidates consolidated   : 995
Active user-channel pairs : 2,281
Eligible Future user-days : 103,395
Active channel-days       : 102,626

BEHAVIORAL ASSESSMENT BY CHANNEL DIVERSITY
----------------------------------------------------------------------------
                      candidates  users_with_any_p95  users_with_simultaneous_p95  users_with_zero_mad  median_multi_channel_days  median_p95_rate_range_pct
active_channel_count                                                                                                                                        
2                            704                 602                           31                  493                       16.0                      2.857
3                            291                 265                           18                  255                       26.0                      5.195

VALIDATION
Candidate population          : PASS
User-channel recon


## 3.9 Numerical & Logical Validation

**Research Question:** Are all Notebook 03 frequency-based findings numerically and logically consistent with the original Future event counts, the existing historical Baseline, and previous notebook outputs?

**Objectives:**
- Verify the 995-candidate population and eligible Future user-days.
- Independently reconcile daily event counts and channel-active days.
- Verify that historical statistics match the existing Baseline.
- Recalculate comparison classifications using the existing reference values.
- Validate per-user frequency deviations and MAD-based consistency.
- Reconcile monthly event counts, active days, and eligible workdays.
- Validate simultaneous cross-channel exceedances.
- Confirm the completeness and uniqueness of consolidated profiles.

**Constraints:**
- Frequency/event counts only.
- No historical Baseline recalculation.
- No modification of source datasets.
- No new attack rules, thresholds, or user exclusions.
- No files written before all checks pass.

**Expected Output:** A numerical and logical validation report showing the status of all 12 checks.


In [11]:

# ============================================================
# 3.9 NUMERICAL & LOGICAL VALIDATION
# ============================================================

import numpy as np
import pandas as pd

audit_results = []

def audit_check(name, condition):
    if not bool(condition):
        raise AssertionError(f"FAILED: {name}")
    audit_results.append(name)

# ------------------------------------------------------------
# 01. CANDIDATE POPULATION
# ------------------------------------------------------------

audit_check(
    "01 Candidate population",
    len(candidate_profile) == 995
    and candidate_profile["user"].is_unique
    and set(candidate_profile["user"]) == candidate_users
    and int((candidate_profile["active_channel_count"] == 2).sum()) == 704
    and int((candidate_profile["active_channel_count"] == 3).sum()) == 291
)

# ------------------------------------------------------------
# 02. FUTURE USER-DAY INTEGRITY
# ------------------------------------------------------------

audit_check(
    "02 Future user-day integrity",
    len(candidate_daily) == 103395
    and candidate_daily["user"].nunique() == 995
    and not candidate_daily.duplicated(["user", "day"]).any()
    and candidate_daily["day"].between(
        "2011-01-01", "2011-06-01"
    ).all()
)

# ------------------------------------------------------------
# 03. DAILY EVENT-COUNT VALIDITY
# ------------------------------------------------------------

count_columns = list(CHANNELS.values())
count_values = candidate_daily[count_columns].to_numpy(
    dtype="float64"
)

audit_check(
    "03 Event-count validity",
    np.isfinite(count_values).all()
    and (count_values >= 0).all()
    and np.equal(count_values, np.floor(count_values)).all()
)

# ------------------------------------------------------------
# 04. ORIGINAL BASELINE REFERENCE MATCH
# ------------------------------------------------------------

baseline_match = True

for channel, metrics in BASELINE_METRICS.items():

    saved = (
        baseline_reference_audit.loc[
            baseline_reference_audit["channel"] == channel.upper()
        ]
        .set_index("user")
        .sort_index()
    )

    original = (
        candidate_baseline
        .set_index("user")
        .loc[saved.index]
    )

    for statistic in ["p50", "p75", "p90", "p95", "mad"]:

        baseline_match &= np.allclose(
            saved[f"baseline_{statistic}"].to_numpy(
                dtype="float64"
            ),
            original[metrics[statistic]].to_numpy(
                dtype="float64"
            ),
            equal_nan=True,
            rtol=0,
            atol=0
        )

audit_check(
    "04 Existing Baseline values",
    baseline_match
    and len(baseline_reference_audit) == 2985
)

# ------------------------------------------------------------
# 05. INDEPENDENT DAILY CHANNEL-COUNT RECONCILIATION
# ------------------------------------------------------------

metric_to_channel = {
    "external_email_count": "EMAIL",
    "usb_copy_count": "USB",
    "www_upload_count": "HTTP"
}

direct_daily = candidate_daily.melt(
    id_vars=["user", "day"],
    value_vars=list(metric_to_channel),
    var_name="metric",
    value_name="direct_count"
)

direct_daily["channel"] = (
    direct_daily["metric"].map(metric_to_channel)
)

direct_daily = direct_daily.loc[
    direct_daily["direct_count"] > 0,
    ["user", "day", "channel", "direct_count"]
]

comparison_check = direct_daily.merge(
    future_frequency_comparison[
        ["user", "day", "channel", "event_count"]
    ],
    on=["user", "day", "channel"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

audit_check(
    "05 Daily channel-count reconciliation",
    len(comparison_check) == 102626
    and comparison_check["_merge"].eq("both").all()
    and np.array_equal(
        comparison_check["direct_count"].to_numpy(),
        comparison_check["event_count"].to_numpy()
    )
)

# ------------------------------------------------------------
# 06. HISTORICAL FREQUENCY BAND RECONSTRUCTION
# ------------------------------------------------------------

daily = future_frequency_comparison

reconstructed_bands = np.select(
    [
        daily["event_count"] <= daily["baseline_p50"],
        daily["event_count"] <= daily["baseline_p75"],
        daily["event_count"] <= daily["baseline_p90"],
        daily["event_count"] <= daily["baseline_p95"],
        daily["event_count"] > daily["baseline_p95"]
    ],
    [
        "AT_OR_BELOW_P50",
        "P50_TO_P75",
        "P75_TO_P90",
        "P90_TO_P95",
        "ABOVE_P95"
    ],
    default="INVALID"
)

audit_check(
    "06 Frequency-band reconstruction",
    np.array_equal(
        reconstructed_bands,
        daily["frequency_band"].to_numpy()
    )
    and int((reconstructed_bands == "ABOVE_P95").sum()) == 3733
)

# ------------------------------------------------------------
# 07. INDEPENDENT USER-CHANNEL P95 RECONCILIATION
# ------------------------------------------------------------

direct_p95 = (
    daily.assign(
        direct_above_p95=(
            daily["event_count"] > daily["baseline_p95"]
        )
    )
    .groupby(["user", "channel"])["direct_above_p95"]
    .sum()
    .sort_index()
)

saved_p95 = (
    user_channel_deviation_profile
    .set_index(["user", "channel"])["days_above_p95"]
    .sort_index()
)

audit_check(
    "07 User-channel P95 exceedances",
    direct_p95.index.equals(saved_p95.index)
    and np.array_equal(
        direct_p95.to_numpy(),
        saved_p95.to_numpy()
    )
    and int(direct_p95.sum()) == 3733
)

# ------------------------------------------------------------
# 08. INDEPENDENT HISTORICAL P50-GAP RECONCILIATION
# ------------------------------------------------------------

direct_median_gap = (
    daily.assign(
        direct_abs_gap=(
            daily["event_count"] - daily["baseline_p50"]
        ).abs()
    )
    .groupby(["user", "channel"])["direct_abs_gap"]
    .median()
    .sort_index()
)

saved_median_gap = (
    user_channel_consistency
    .set_index(["user", "channel"])[
        "median_absolute_p50_gap"
    ]
    .sort_index()
)

audit_check(
    "08 Historical P50-gap consistency",
    direct_median_gap.index.equals(saved_median_gap.index)
    and np.allclose(
        direct_median_gap.to_numpy(),
        saved_median_gap.to_numpy(),
        rtol=0,
        atol=1e-9
    )
)

# ------------------------------------------------------------
# 09. HISTORICAL ZERO-MAD HANDLING
# ------------------------------------------------------------

consistency = user_channel_consistency

zero_mad = consistency["baseline_mad"].eq(0)
positive_mad = consistency["baseline_mad"].gt(0)

audit_check(
    "09 Zero-MAD handling",
    consistency.loc[
        zero_mad, "relative_median_gap_mad"
    ].isna().all()
    and consistency.loc[
        positive_mad, "relative_median_gap_mad"
    ].notna().all()
    and consistency["baseline_mad"].ge(0).all()
)

# ------------------------------------------------------------
# 10. MONTHLY EVENT AND WORKDAY RECONCILIATION
# ------------------------------------------------------------

direct_monthly = (
    daily.assign(month=daily["day"].dt.to_period("M"))
    .groupby(["user", "channel", "month"])
    .agg(
        direct_active_days=("event_count", "size"),
        direct_total_events=("event_count", "sum")
    )
)

monthly_check = monthly_frequency.set_index(
    ["user", "channel", "month"]
).join(
    direct_monthly,
    how="left"
)

direct_eligible_months = (
    candidate_daily.assign(
        month=candidate_daily["day"].dt.to_period("M")
    )
    .groupby(["user", "month"])
    .size()
    .rename("direct_eligible_workdays")
)

monthly_check = monthly_check.join(
    direct_eligible_months,
    on=["user", "month"],
    how="left"
)

monthly_valid = (
    direct_monthly.index.isin(monthly_check.index).all()
    and monthly_check["direct_eligible_workdays"].notna().all()
    and np.array_equal(
        monthly_check["eligible_workdays"].to_numpy(),
        monthly_check["direct_eligible_workdays"].to_numpy()
    )
    and np.array_equal(
        monthly_check["active_days"].to_numpy(),
        monthly_check["direct_active_days"]
        .fillna(0).to_numpy()
    )
    and np.allclose(
        monthly_check["total_events"].to_numpy(
            dtype="float64"
        ),
        monthly_check["direct_total_events"]
        .fillna(0).to_numpy(dtype="float64"),
        rtol=0,
        atol=1e-9
    )
)

audit_check(
    "10 Monthly frequency reconciliation",
    monthly_valid
    and not monthly_full["month"].eq(
        pd.Period("2011-06", freq="M")
    ).any()
)

# ------------------------------------------------------------
# 11. INDEPENDENT SIMULTANEOUS P95 RECONSTRUCTION
# ------------------------------------------------------------

daily_p95_pivot = (
    daily.assign(
        exceeds_p95=(
            daily["event_count"] > daily["baseline_p95"]
        )
    )
    .pivot(
        index=["user", "day"],
        columns="channel",
        values="exceeds_p95"
    )
)

direct_multichannel = pd.DataFrame({
    "active_channels": daily_p95_pivot.notna().sum(axis=1),
    "p95_exceeding_channels": daily_p95_pivot.eq(True).sum(
        axis=1
    )
})

direct_multichannel = direct_multichannel.loc[
    direct_multichannel["active_channels"] >= 2
]

direct_simultaneous_days = int((
    direct_multichannel["p95_exceeding_channels"] >= 2
).sum())

direct_all_exceed_days = int((
    direct_multichannel["p95_exceeding_channels"]
    == direct_multichannel["active_channels"]
).sum())

audit_check(
    "11 Simultaneous-channel P95 validation",
    len(direct_multichannel) == 24625
    and direct_simultaneous_days == 51
    and direct_all_exceed_days == 43
)

# ------------------------------------------------------------
# 12. CONSOLIDATED PROFILE RECONCILIATION
# ------------------------------------------------------------

final_user = candidate_behavior_profile
final_channel = candidate_channel_behavior_profile

audit_check(
    "12 Final profile consistency",
    len(final_user) == 995
    and final_user["user"].is_unique
    and len(final_channel) == 2281
    and not final_channel.duplicated(
        ["user", "channel"]
    ).any()
    and int(final_user["future_workdays"].sum()) == 103395
    and int(final_user["active_channel_days"].sum()) == 102626
    and int(final_user[
        "natural_p95_exceedance_channel_days"
    ].sum()) == 3733
    and int(final_user[
        "days_with_simultaneous_p95"
    ].sum()) == 51
    and set(final_user["user"]) == candidate_users
)

# ------------------------------------------------------------
# FINAL REPORT — NO FILES WRITTEN
# ------------------------------------------------------------

print("=" * 72)
print("NOTEBOOK 03 — NUMERICAL & LOGICAL VALIDATION")
print("=" * 72)

for name in audit_results:
    print(f"PASS — {name}")

print("-" * 72)
print(f"RESULT: {len(audit_results)}/12 CHECKS PASSED")
print("Historical Baseline recalculated : NO")
print("Source datasets modified        : NO")
print("Files written                   : NONE")


NOTEBOOK 03 — NUMERICAL & LOGICAL VALIDATION
PASS — 01 Candidate population
PASS — 02 Future user-day integrity
PASS — 03 Event-count validity
PASS — 04 Existing Baseline values
PASS — 05 Daily channel-count reconciliation
PASS — 06 Frequency-band reconstruction
PASS — 07 User-channel P95 exceedances
PASS — 08 Historical P50-gap consistency
PASS — 09 Zero-MAD handling
PASS — 10 Monthly frequency reconciliation
PASS — 11 Simultaneous-channel P95 validation
PASS — 12 Final profile consistency
------------------------------------------------------------------------
RESULT: 12/12 CHECKS PASSED
Historical Baseline recalculated : NO
Source datasets modified        : NO
Files written                   : NONE



## 3.10 Final Consolidation & Documentation

### 3.10.1 Validated Profile Export

**Research Question:** Can the validated behavioral results be saved without losing records, changing values, or duplicating user-channel keys?

**Objectives:**
- Export the consolidated user-channel frequency profile.
- Export the consolidated candidate-level behavioral profile.
- Read both files back and verify their contents.
- Preserve all 995 candidates and 2,281 active user-channel pairs.

**Constraints:**
- Event-count frequency analysis only.
- No Baseline recalculation or source modification.
- No intermediate analysis files.
- Export only after the 12 numerical and logical checks have passed.

**Expected Output:** Two verified Parquet files under `reports/03_Channel_Behavior_Audit/`.


In [12]:

# ============================================================
# 3.10.1 VALIDATED PROFILE EXPORT
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. REQUIRE COMPLETED VALIDATION
# ------------------------------------------------------------

assert len(audit_results) == 12, (
    "Complete Section 3.9 before exporting."
)

assert len(candidate_behavior_profile) == 995
assert len(candidate_channel_behavior_profile) == 2281

# ------------------------------------------------------------
# 2. PREPARE FINAL PROFILES
# ------------------------------------------------------------

final_user_profile = (
    candidate_behavior_profile
    .sort_values("user")
    .reset_index(drop=True)
    .copy()
)

final_channel_profile = (
    candidate_channel_behavior_profile
    .sort_values(["user", "channel"])
    .reset_index(drop=True)
    .copy()
)

assert final_user_profile["user"].is_unique

assert not final_channel_profile.duplicated(
    ["user", "channel"]
).any()

assert set(final_user_profile["user"]) == candidate_users

assert set(final_channel_profile["user"]) == candidate_users

# ------------------------------------------------------------
# 3. OUTPUT PATHS
# ------------------------------------------------------------

REPORT_DIR_03.mkdir(
    parents=True,
    exist_ok=True
)

USER_OUTPUT = (
    REPORT_DIR_03
    / "candidate_behavior_profile.parquet"
)

CHANNEL_OUTPUT = (
    REPORT_DIR_03
    / "user_channel_frequency_comparison.parquet"
)

# Prevent accidental replacement of any input dataset.
source_paths = {
    PROFILE_PATH.resolve(),
    FUTURE_PATH.resolve(),
    BASELINE_PATH.resolve()
}

assert USER_OUTPUT.resolve() not in source_paths
assert CHANNEL_OUTPUT.resolve() not in source_paths
assert USER_OUTPUT.resolve() != CHANNEL_OUTPUT.resolve()

# ------------------------------------------------------------
# 4. EXPORT CONSOLIDATED RESULTS
# ------------------------------------------------------------

final_user_profile.to_parquet(
    USER_OUTPUT,
    index=False,
    compression="zstd"
)

final_channel_profile.to_parquet(
    CHANNEL_OUTPUT,
    index=False,
    compression="zstd"
)

# ------------------------------------------------------------
# 5. READ-BACK VERIFICATION
# ------------------------------------------------------------

saved_user = pd.read_parquet(USER_OUTPUT)
saved_channel = pd.read_parquet(CHANNEL_OUTPUT)

pd.testing.assert_frame_equal(
    final_user_profile,
    saved_user,
    check_dtype=True,
    check_exact=True
)

pd.testing.assert_frame_equal(
    final_channel_profile,
    saved_channel,
    check_dtype=True,
    check_exact=True
)

assert saved_user["user"].nunique() == 995

assert not saved_user["user"].duplicated().any()

assert not saved_channel.duplicated(
    ["user", "channel"]
).any()

assert len(saved_channel) == 2281

assert int(saved_user["future_workdays"].sum()) == 103395

assert int(saved_user["active_channel_days"].sum()) == 102626

assert int(
    saved_user["natural_p95_exceedance_channel_days"].sum()
) == 3733

assert int(
    saved_user["days_with_simultaneous_p95"].sum()
) == 51

# ------------------------------------------------------------
# 6. CONCISE EXPORT REPORT
# ------------------------------------------------------------

print("=" * 72)
print("NOTEBOOK 03 — VALIDATED PROFILE EXPORT")
print("=" * 72)

print(f"Previous audit       : {len(audit_results)}/12 PASS")

print("\nSAVED FILES")
print("-" * 72)

for path, df in [
    (USER_OUTPUT, saved_user),
    (CHANNEL_OUTPUT, saved_channel)
]:
    print(
        f"{path.name} | "
        f"{len(df):,} rows x {len(df.columns)} columns"
    )

print("\nVALIDATION")
print("User population       : PASS")
print("User-channel coverage : PASS")
print("Parquet read-back     : PASS")
print("Exact value comparison: PASS")
print("Historical Baseline   : UNCHANGED")
print("Original Future       : UNCHANGED")
print("EXPORT STATUS         : PASS")

print("\nOutput directory:")
print(REPORT_DIR_03)


NOTEBOOK 03 — VALIDATED PROFILE EXPORT
Previous audit       : 12/12 PASS

SAVED FILES
------------------------------------------------------------------------
candidate_behavior_profile.parquet | 995 rows x 20 columns
user_channel_frequency_comparison.parquet | 2,281 rows x 27 columns

VALIDATION
User population       : PASS
User-channel coverage : PASS
Parquet read-back     : PASS
Exact value comparison: PASS
Historical Baseline   : UNCHANGED
Original Future       : UNCHANGED
EXPORT STATUS         : PASS

Output directory:
C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\03_Channel_Behavior_Audit



### 3.10.2 Data Dictionary & Final Summary

**Research Question:** Are all exported behavioral features documented and ready for use in subsequent simulation-feasibility analyses?

**Objectives:**
- Document every column in the two validated Parquet files.
- Record data types, definitions, sources, and null-value meanings.
- Produce a concise summary of the main frequency-based findings.
- Verify that the documentation matches the exported files.

**Constraints:**
- No additional behavioral analysis.
- No Baseline recalculation.
- No changes to the original datasets.
- No extra intermediate files.

**Expected Output:**
- `channel_behavior_data_dictionary.csv`
- `channel_behavior_summary.md`


In [13]:

# ============================================================
# 3.10.2 DATA DICTIONARY & FINAL SUMMARY
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. READ VERIFIED FINAL FILES
# ------------------------------------------------------------

assert len(audit_results) == 12

user_df = pd.read_parquet(USER_OUTPUT)
channel_df = pd.read_parquet(CHANNEL_OUTPUT)

assert user_df.shape == (995, 20)
assert channel_df.shape == (2281, 27)

# ------------------------------------------------------------
# 2. EXACT COLUMN DEFINITIONS
# ------------------------------------------------------------

# Format: column -> (definition, source)

DEFINITIONS = {
    "user": (
        "Unique user identifier.",
        "Notebook 01 / original data"
    ),
    "future_workdays": (
        "Number of eligible Future workdays for the user.",
        "Notebook 01"
    ),
    "active_channel_count": (
        "Number of naturally active target channels in Future.",
        "Notebook 01"
    ),
    "observed_channel_set": (
        "Set of naturally observed target channels in Future.",
        "Notebook 01"
    ),
    "naturally_active_channels": (
        "Number of naturally active channels represented in the frequency profile.",
        "Section 3.7"
    ),
    "min_p95_rate_pct": (
        "Lowest natural P95 exceedance percentage among the user's active channels.",
        "Section 3.7"
    ),
    "max_p95_rate_pct": (
        "Highest natural P95 exceedance percentage among the user's active channels.",
        "Section 3.7"
    ),
    "channels_with_any_p95_exceedance": (
        "Number of channels with at least one natural Future day above historical P95.",
        "Section 3.7"
    ),
    "p95_rate_range_pct": (
        "Difference between maximum and minimum channel P95 exceedance rates, in percentage points.",
        "Section 3.7"
    ),
    "active_channel_days": (
        "Sum of channel-active Future days across the user's target channels; a calendar date may contribute more than once.",
        "Section 3.8"
    ),
    "natural_p95_exceedance_channel_days": (
        "Sum of natural channel-days exceeding historical P95 across the user's channels.",
        "Sections 3.4 and 3.8"
    ),
    "channels_with_zero_mad": (
        "Number of active channels whose existing historical MAD equals zero.",
        "Sections 3.5 and 3.8"
    ),
    "channels_with_median_gap_above_mad": (
        "Number of active channels with positive historical MAD where the median absolute Future-to-P50 gap exceeds MAD.",
        "Sections 3.5 and 3.8"
    ),
    "multi_channel_user_days": (
        "Number of eligible Future days with at least two naturally active target channels.",
        "Sections 3.7 and 3.8"
    ),
    "multi_channel_days_with_any_p95": (
        "Number of naturally multi-channel days where at least one active channel exceeds historical P95.",
        "Sections 3.7 and 3.8"
    ),
    "days_with_simultaneous_p95": (
        "Number of days where at least two naturally active channels simultaneously exceed their own historical P95.",
        "Sections 3.7 and 3.8"
    ),
    "days_all_active_above_p95": (
        "Number of naturally multi-channel days where all active target channels exceed their own historical P95.",
        "Sections 3.7 and 3.8"
    ),
    "has_any_p95_exceedance": (
        "True if at least one natural Future channel-day exceeds historical P95.",
        "Section 3.8"
    ),
    "has_simultaneous_p95": (
        "True if at least one Future day has simultaneous P95 exceedances in two or more active channels.",
        "Section 3.8"
    ),
    "has_zero_mad_channel": (
        "True if any naturally active channel has historical MAD equal to zero.",
        "Section 3.8"
    ),
    "channel": (
        "Target activity channel: EMAIL, USB, or HTTP.",
        "Sections 3.1 and 3.2"
    ),
    "future_active_days": (
        "Number of eligible Future workdays with positive natural event count in this channel.",
        "Notebook 01 / Section 3.2"
    ),
    "baseline_active_days": (
        "Number of historical channel-active workdays recorded in the existing Baseline.",
        "Existing Baseline, read-only"
    ),
    "baseline_p50": (
        "Existing historical P50 of event counts on channel-active days.",
        "Existing Baseline, read-only"
    ),
    "baseline_p75": (
        "Existing historical P75 of event counts on channel-active days.",
        "Existing Baseline, read-only"
    ),
    "baseline_p90": (
        "Existing historical P90 of event counts on channel-active days.",
        "Existing Baseline, read-only"
    ),
    "baseline_p95": (
        "Existing historical P95 of event counts on channel-active days.",
        "Existing Baseline, read-only"
    ),
    "baseline_mad": (
        "Existing historical median absolute deviation of event counts on channel-active days.",
        "Existing Baseline, read-only"
    ),
    "has_baseline_reference": (
        "Whether the existing channel-specific historical frequency reference passes the availability checks.",
        "Section 3.2 / Notebook 01"
    ),
    "days_above_p90": (
        "Number of natural channel-active Future days with event count above historical P90.",
        "Section 3.4"
    ),
    "days_above_p95": (
        "Number of natural channel-active Future days with event count above historical P95.",
        "Section 3.4"
    ),
    "p95_exceedance_rate_pct": (
        "100 times days_above_p95 divided by future_active_days.",
        "Section 3.4"
    ),
    "has_p95_exceedance": (
        "True if days_above_p95 is greater than zero.",
        "Section 3.4"
    ),
    "total_positive_p95_difference": (
        "Sum of max(event_count - historical P95, 0) across natural Future channel-active days; not injection capacity.",
        "Section 3.4"
    ),
    "max_positive_p95_difference": (
        "Maximum of max(event_count - historical P95, 0) across natural Future channel-active days.",
        "Section 3.4"
    ),
    "median_signed_p50_gap": (
        "Median of (Future event count - historical P50) on channel-active Future days.",
        "Section 3.5"
    ),
    "median_absolute_p50_gap": (
        "Median absolute difference between Future event counts and historical P50.",
        "Section 3.5"
    ),
    "historical_mad_zero": (
        "True when existing historical MAD equals zero.",
        "Section 3.5"
    ),
    "relative_median_gap_mad": (
        "Median absolute Future-to-P50 gap divided by historical MAD, defined only for MAD greater than zero.",
        "Section 3.5"
    ),
    "median_gap_exceeds_historical_mad": (
        "True when historical MAD is positive and the median absolute Future-to-P50 gap exceeds it.",
        "Section 3.5"
    ),
    "zero_mad_with_future_difference": (
        "True when historical MAD equals zero and at least one Future active day differs from historical P50.",
        "Section 3.5"
    ),
    "frequency_direction": (
        "Direction of the median signed Future-to-P50 gap: above, below, or equal.",
        "Section 3.5"
    ),
    "months_observed": (
        "Number of January-May 2011 months with eligible Future workdays for the user-channel profile.",
        "Section 3.6"
    ),
    "months_with_activity": (
        "Number of January-May 2011 months with positive natural event counts in the channel.",
        "Section 3.6"
    ),
    "eligible_day_frequency_range": (
        "Maximum minus minimum monthly events per eligible workday across January-May 2011; defined for at least two observed months.",
        "Section 3.6"
    ),
    "active_day_frequency_range": (
        "Maximum minus minimum monthly events per channel-active day across January-May 2011; defined for at least two active months.",
        "Section 3.6"
    )
}

NULL_MEANINGS = {
    "relative_median_gap_mad":
        "Undefined when historical MAD equals zero.",
    "eligible_day_frequency_range":
        "Undefined when fewer than two monthly observations exist.",
    "active_day_frequency_range":
        "Undefined when fewer than two months have channel activity."
}

# ------------------------------------------------------------
# 3. GENERATE DICTIONARY FROM ACTUAL SAVED SCHEMAS
# ------------------------------------------------------------

dictionary_rows = []

datasets = [
    (
        USER_OUTPUT.name,
        "One row per candidate user",
        user_df
    ),
    (
        CHANNEL_OUTPUT.name,
        "One row per naturally active user-channel pair",
        channel_df
    )
]

for filename, grain, df in datasets:

    assert df.columns.is_unique

    for position, column in enumerate(df.columns, start=1):

        assert column in DEFINITIONS, (
            f"Missing definition: {filename} / {column}"
        )

        definition, source = DEFINITIONS[column]
        null_count = int(df[column].isna().sum())

        if null_count > 0:
            assert column in NULL_MEANINGS, (
                f"Unexpected undocumented nulls: "
                f"{filename} / {column}"
            )

        dictionary_rows.append({
            "dataset": filename,
            "grain": grain,
            "column_position": position,
            "column": column,
            "dtype": str(df[column].dtype),
            "null_count": null_count,
            "definition": definition,
            "source": source,
            "null_meaning": NULL_MEANINGS.get(
                column,
                "No null values expected."
            )
        })

data_dictionary_03 = pd.DataFrame(dictionary_rows)

assert len(data_dictionary_03) == 47

assert not data_dictionary_03.duplicated(
    ["dataset", "column"]
).any()

assert data_dictionary_03[
    ["definition", "source", "null_meaning"]
].notna().all().all()

# ------------------------------------------------------------
# 4. SAVE AND VERIFY DATA DICTIONARY
# ------------------------------------------------------------

DICTIONARY_OUTPUT = (
    REPORT_DIR_03
    / "channel_behavior_data_dictionary.csv"
)

data_dictionary_03.to_csv(
    DICTIONARY_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

saved_dictionary = pd.read_csv(
    DICTIONARY_OUTPUT,
    encoding="utf-8-sig",
    keep_default_na=False
)

assert len(saved_dictionary) == 47

assert (
    saved_dictionary.groupby("dataset").size().to_dict()
    == {
        USER_OUTPUT.name: 20,
        CHANNEL_OUTPUT.name: 27
    }
)

assert not saved_dictionary[
    ["dataset", "column"]
].duplicated().any()

# ------------------------------------------------------------
# 5. COMPOSE FINAL SUMMARY FROM SAVED RESULTS
# ------------------------------------------------------------

users_with_p95 = int(
    user_df["has_any_p95_exceedance"].sum()
)

users_with_simultaneous = int(
    user_df["has_simultaneous_p95"].sum()
)

users_with_zero_mad = int(
    user_df["has_zero_mad_channel"].sum()
)

summary_lines = [
    "# MUBSIR — Notebook 03: Channel Behavior Audit",
    "",
    "## Scope",
    "",
    "- Candidates: 995 naturally multi-channel users.",
    "- Future: 2011-01-03 to 2011-06-01.",
    "- Historical Baseline: existing read-only frequency reference.",
    "- Analysis: natural event counts only.",
    "- No injection or scenario assignment.",
    "",
    "## Population and Coverage",
    "",
    f"- Candidate users: {len(user_df):,}.",
    f"- Active user-channel pairs: {len(channel_df):,}.",
    f"- Eligible Future user-days: "
    f"{int(user_df['future_workdays'].sum()):,}.",
    f"- Active Future channel-days: "
    f"{int(user_df['active_channel_days'].sum()):,}.",
    "",
    "## Natural Frequency Deviations",
    "",
    f"- Natural channel-days above historical P95: "
    f"{int(channel_df['days_above_p95'].sum()):,}.",
    f"- Users with at least one natural P95 exceedance: "
    f"{users_with_p95:,}.",
    f"- Users with a simultaneous multi-channel P95 exceedance: "
    f"{users_with_simultaneous:,}.",
    f"- Multi-channel days with simultaneous P95 exceedances: "
    f"{int(user_df['days_with_simultaneous_p95'].sum()):,}.",
    f"- Users with at least one zero-MAD historical channel: "
    f"{users_with_zero_mad:,}.",
    "",
    "## Channel-Level Results",
    "",
    "| Channel | Active users | Active days | Days above P95 |",
    "|---|---:|---:|---:|"
]

for channel in ["EMAIL", "USB", "HTTP"]:

    part = channel_df.loc[
        channel_df["channel"] == channel
    ]

    summary_lines.append(
        f"| {channel} | {len(part):,} | "
        f"{int(part['future_active_days'].sum()):,} | "
        f"{int(part['days_above_p95'].sum()):,} |"
    )

summary_lines.extend([
    "",
    "## Main Findings",
    "",
    "1. All naturally active Future user-channel pairs "
    "have usable historical frequency references.",
    "2. Natural Future activity can exceed historical P95 "
    "without any injected attack.",
    "3. Simultaneous natural P95 exceedances across channels "
    "are much less frequent than individual exceedances.",
    "4. Zero historical MAD is common and requires explicit "
    "handling in future feasibility analysis.",
    "5. Average monthly event frequency can remain relatively "
    "similar while individual users vary.",
    "",
    "## Methodological Boundaries",
    "",
    "- Historical Baseline was not recalculated or modified.",
    "- Original Future data was not modified.",
    "- P95 exceedance is not an attack label.",
    "- Natural frequency differences are not approved "
    "injection budgets.",
    "- Final injection feasibility and stealth rules "
    "have not yet been established.",
    "",
    "## Quality Assurance",
    "",
    "- Numerical and logical validation: 12/12 PASS.",
    "- Parquet files exported and read back successfully.",
    "- Data dictionary: 47/47 columns documented.",
    "",
    "## Final Deliverables",
    "",
    f"- `{USER_OUTPUT.name}`",
    f"- `{CHANNEL_OUTPUT.name}`",
    f"- `{DICTIONARY_OUTPUT.name}`",
    "- `channel_behavior_summary.md`",
    ""
])

SUMMARY_OUTPUT = (
    REPORT_DIR_03
    / "channel_behavior_summary.md"
)

summary_text = "\n".join(summary_lines)

SUMMARY_OUTPUT.write_text(
    summary_text,
    encoding="utf-8"
)

assert SUMMARY_OUTPUT.read_text(
    encoding="utf-8"
) == summary_text

# ------------------------------------------------------------
# 6. FINAL NOTEBOOK STATUS
# ------------------------------------------------------------

print("=" * 70)
print("NOTEBOOK 03 — FINAL DOCUMENTATION")
print("=" * 70)

print(f"Candidate users          : {len(user_df):,}")
print(f"User-channel records     : {len(channel_df):,}")
print(f"Previous validation      : {len(audit_results)}/12 PASS")
print(f"Dictionary coverage      : {len(data_dictionary_03)}/47 PASS")

print("\nFINAL DELIVERABLES")
print("-" * 70)

for path in [
    USER_OUTPUT,
    CHANNEL_OUTPUT,
    DICTIONARY_OUTPUT,
    SUMMARY_OUTPUT
]:
    print(path.name)

print("\nVALIDATION")
print("Saved schema coverage    : PASS")
print("Dictionary read-back     : PASS")
print("Summary read-back        : PASS")
print("Historical Baseline      : UNCHANGED")
print("Original Future          : UNCHANGED")
print("NOTEBOOK 03 STATUS       : COMPLETE")


NOTEBOOK 03 — FINAL DOCUMENTATION
Candidate users          : 995
User-channel records     : 2,281
Previous validation      : 12/12 PASS
Dictionary coverage      : 47/47 PASS

FINAL DELIVERABLES
----------------------------------------------------------------------
candidate_behavior_profile.parquet
user_channel_frequency_comparison.parquet
channel_behavior_data_dictionary.csv
channel_behavior_summary.md

VALIDATION
Saved schema coverage    : PASS
Dictionary read-back     : PASS
Summary read-back        : PASS
Historical Baseline      : UNCHANGED
Original Future          : UNCHANGED
NOTEBOOK 03 STATUS       : COMPLETE


توحيد الملخصات

In [14]:

# ============================================================
# MUBSIR — UNIFIED NOTEBOOK REPORT SUMMARIES (00–03)
# ============================================================

from pathlib import Path

REPORTS_ROOT = PROJECT_ROOT / "reports"

SPECS = {
    "00_Data_Inventory_and_Integrity": {
        "title": "Data Inventory & Integrity",
        "scope": "All original datasets; Baseline and Future integrity.",
        "findings": [
            "3,995 benign users with historical Baseline.",
            "3,737 users with eligible Future activity.",
            "387,162 eligible Future feature records.",
            "Processed-data integrity and source reconciliation completed."
        ],
        "validation": "13/13 PASS",
        "legacy": "00_data_integrity_summary.md",
        "required": ["00_final_qa_summary.csv"]
    },
    "01_User_Population_Audit": {
        "title": "User Population Audit",
        "scope": "All 3,995 users; natural Future behavior and channel use.",
        "findings": [
            "3,737 users with eligible Future workdays.",
            "2,742 naturally single-channel users.",
            "704 naturally two-channel users.",
            "291 naturally three-channel users.",
            "995 preliminary multi-channel candidates."
        ],
        "validation": "Completed population/profile validation.",
        "legacy": "population_audit_summary.md",
        "required": [
            "user_population_profile.parquet",
            "user_population_profile_dictionary.csv"
        ]
    },
    "02_Temporal_Window_Audit": {
        "title": "Temporal Window Audit",
        "scope": "Eligible Future workdays; natural channel rotation.",
        "findings": [
            "20,436,421 overlapping candidate window placements.",
            "Window lengths examined: 1–108 eligible workdays.",
            "995 preliminary multi-channel candidates.",
            "Ordered rotation, recurrence, and cadence evaluated.",
            "Long rotations reconstructed from natural channel-day activity."
        ],
        "validation": "12/12 PASS; 168/168 documented columns.",
        "legacy": None,
        "required": [
            "user_window_length_audit.parquet",
            "rotation_feasibility_audit.parquet",
            "notebook02_data_dictionary.csv"
        ]
    },
    "03_Channel_Behavior_Audit": {
        "title": "Channel Behavior Audit",
        "scope": "995 multi-channel candidates; natural Future event counts.",
        "findings": [
            "103,395 eligible Future user-days.",
            "2,281 naturally active user-channel pairs.",
            "102,626 active Future channel-days.",
            "3,733 natural channel-days exceeded historical P95.",
            "51 multi-channel days had simultaneous P95 exceedances.",
            "Historical Baseline statistics were used without recalculation."
        ],
        "validation": "12/12 PASS; 47/47 documented columns.",
        "legacy": "channel_behavior_summary.md",
        "required": [
            "candidate_behavior_profile.parquet",
            "user_channel_frequency_comparison.parquet",
            "channel_behavior_data_dictionary.csv"
        ]
    }
}

# ------------------------------------------------------------
# 1. VERIFY ALL REQUIRED REPORTS BEFORE CHANGING ANYTHING
# ------------------------------------------------------------

for notebook, spec in SPECS.items():
    folder = REPORTS_ROOT / notebook
    assert folder.is_dir(), f"Missing report folder: {folder}"

    for filename in spec["required"]:
        assert (folder / filename).is_file(), (
            f"Missing output: {folder / filename}"
        )

    legacy = spec["legacy"]
    canonical = folder / "notebook_summary.md"

    if legacy is not None:
        assert (folder / legacy).is_file() or canonical.is_file(), (
            f"Missing Markdown summary: {notebook}"
        )

print("PREFLIGHT: PASS — All four report folders verified.")

# ------------------------------------------------------------
# 2. GENERATE CONSISTENT MARKDOWN SUMMARIES
# ------------------------------------------------------------

for notebook, spec in SPECS.items():

    folder = REPORTS_ROOT / notebook
    output = folder / "notebook_summary.md"

    legacy = (
        folder / spec["legacy"]
        if spec["legacy"] is not None
        else None
    )

    # Idempotent: don't duplicate the original report
    # when the standardization cell is rerun.
    if output.is_file() and (
        legacy is None or not legacy.is_file()
    ):
        print(f"ALREADY STANDARDIZED: {notebook}")
        continue

    original_content = (
        legacy.read_text(encoding="utf-8")
        if legacy is not None and legacy.is_file()
        else ""
    )

    # Detect final deliverables from actual report files.
    deliverables = sorted(
        p.name for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in {".csv", ".parquet"}
    )

    lines = [
        f"# MUBSIR — {notebook}",
        "",
        "## 1. Objective",
        "",
        spec["title"],
        "",
        "## 2. Data Scope",
        "",
        spec["scope"],
        "",
        "## 3. Key Findings",
        ""
    ]

    lines.extend(f"- {item}" for item in spec["findings"])

    lines.extend([
        "",
        "## 4. Validation",
        "",
        f"- Status: {spec['validation']}",
        "- Original source datasets remain unchanged.",
        "",
        "## 5. Final Deliverables",
        ""
    ])

    lines.extend(f"- `{name}`" for name in deliverables)

    lines.extend([
        "",
        "## 6. Limitations & Next Step",
        "",
        "- Results describe completed analyses only.",
        "- Attack injection feasibility is not yet proven.",
        "- Final attack assignment and injection remain pending.",
        "- Historical Baseline must remain immutable.",
        ""
    ])

    if original_content:
        lines.extend([
            "## 7. Original Detailed Report",
            "",
            "The original report is preserved below.",
            "",
            "---",
            "",
            original_content,
            ""
        ])

    unified_text = "\n".join(lines)

    output.write_text(unified_text, encoding="utf-8")

    # Verify the complete original report was preserved.
    saved_text = output.read_text(encoding="utf-8")

    assert saved_text == unified_text

    if original_content:
        assert original_content in saved_text

    # Remove the old filename only after verifying the
    # full content is preserved in notebook_summary.md.
    if legacy is not None and legacy != output:
        if legacy.is_file():
            legacy.unlink()

    print(f"PASS: {notebook}/notebook_summary.md")

# ------------------------------------------------------------
# 3. FINAL VERIFICATION
# ------------------------------------------------------------

for notebook in SPECS:
    path = REPORTS_ROOT / notebook / "notebook_summary.md"

    assert path.is_file()
    content = path.read_text(encoding="utf-8")

    for section in [
        "## 1. Objective",
        "## 2. Data Scope",
        "## 3. Key Findings",
        "## 4. Validation",
        "## 5. Final Deliverables",
        "## 6. Limitations & Next Step"
    ]:
        assert section in content

print("\n" + "=" * 65)
print("MUBSIR — UNIFIED REPORT DOCUMENTATION")
print("=" * 65)
print("Notebooks standardized : 4/4")
print("Summary filename       : notebook_summary.md")
print("Original report text   : PRESERVED")
print("Parquet/CSV files      : UNCHANGED")
print("Baseline/Future        : UNCHANGED")
print("STATUS                 : PASS")


PREFLIGHT: PASS — All four report folders verified.
PASS: 00_Data_Inventory_and_Integrity/notebook_summary.md
PASS: 01_User_Population_Audit/notebook_summary.md
PASS: 02_Temporal_Window_Audit/notebook_summary.md
PASS: 03_Channel_Behavior_Audit/notebook_summary.md

MUBSIR — UNIFIED REPORT DOCUMENTATION
Notebooks standardized : 4/4
Summary filename       : notebook_summary.md
Original report text   : PRESERVED
Parquet/CSV files      : UNCHANGED
Baseline/Future        : UNCHANGED
STATUS                 : PASS
